In [ ]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

In [ ]:
BEN_DIR = Path("Ben Annotations")
DASH_DIR = Path("Dash Annotations")

In [ ]:

def text_id_from_stem(stem: str):
    # Works for:
    # 11_manual_sample_annotations
    # 11_manual_sample_annotations (1)
    # 514_manual_sample_3_annotations (4)
    m = re.search(r"^(\d+)_manual_sample", stem)
    return m.group(1) if m else None

def norm_text(s: str) -> str:
    # Collapse whitespace so small formatting differences don't break matches
    return " ".join((s or "").split()).strip()

def load_spans(path: Path):
    obj = json.loads(path.read_text(encoding="utf-8"))
    spans = obj.get("spans", [])
    rows = []
    for sp in spans:
        rows.append({
            "text_key": norm_text(sp.get("text", "")),
            "labels_key": tuple(sorted(sp.get("labels", []))),
            "speaker": (sp.get("speaker") or "").strip(),
            "start": sp.get("start", sp.get("start_byte")),
            "end": sp.get("end", sp.get("end_byte")),
            "raw_text": sp.get("text", "")
        })
    return rows

In [ ]:


def summarize_pair(ben_file: Path, dash_file: Path):
    ben = load_spans(ben_file)
    dash = load_spans(dash_file)

    # ---------- Step 1: same tagged texts (ignore labels) ----------
    ben_text_ctr = Counter(r["text_key"] for r in ben)
    dash_text_ctr = Counter(r["text_key"] for r in dash)

    shared_text_ctr = ben_text_ctr & dash_text_ctr
    union_text_ctr = ben_text_ctr | dash_text_ctr
    ben_only_text_ctr = ben_text_ctr - dash_text_ctr
    dash_only_text_ctr = dash_text_ctr - ben_text_ctr

    n_shared_text = sum(shared_text_ctr.values())
    n_union_text = sum(union_text_ctr.values())

    pct_same_texts = (100 * n_shared_text / n_union_text) if n_union_text else 100.0

    # ---------- Step 2: same labels, conditional on shared texts ----------
    ben_text_to_labels = defaultdict(Counter)
    dash_text_to_labels = defaultdict(Counter)

    for r in ben:
        ben_text_to_labels[r["text_key"]][r["labels_key"]] += 1
    for r in dash:
        dash_text_to_labels[r["text_key"]][r["labels_key"]] += 1

    matched_label_instances = 0
    shared_label_comparison_instances = 0
    label_mismatch_examples = []

    for text_key, shared_n in shared_text_ctr.items():
        # How many shared instances got the same labels?
        ben_lab_ctr = ben_text_to_labels[text_key]
        dash_lab_ctr = dash_text_to_labels[text_key]
        same_label_n = sum((ben_lab_ctr & dash_lab_ctr).values())

        matched_label_instances += same_label_n
        shared_label_comparison_instances += shared_n

        # Capture asymmetric label differences for this shared text
        dash_extra = dash_lab_ctr - ben_lab_ctr
        ben_extra = ben_lab_ctr - dash_lab_ctr
        if dash_extra or ben_extra:
            label_mismatch_examples.append({
                "text": text_key[:180],
                "dash_extra_labels": list(dash_extra.elements()),
                "ben_extra_labels": list(ben_extra.elements()),
            })

    pct_label_match_given_shared_text = (
        100 * matched_label_instances / shared_label_comparison_instances
        if shared_label_comparison_instances else 100.0
    )

    # Optional strict metric (text + labels together)
    ben_strict_ctr = Counter((r["text_key"], r["labels_key"]) for r in ben)
    dash_strict_ctr = Counter((r["text_key"], r["labels_key"]) for r in dash)
    strict_overlap = sum((ben_strict_ctr & dash_strict_ctr).values())
    strict_union = sum((ben_strict_ctr | dash_strict_ctr).values())
    pct_strict_overall = (100 * strict_overlap / strict_union) if strict_union else 100.0

    return {
        "ben_file": ben_file.name,
        "dash_file": dash_file.name,

        # Step 1 outputs
        "pct_same_tagged_texts": pct_same_texts,
        "shared_text_count": n_shared_text,
        "union_text_count": n_union_text,
        "dash_only_text_count": sum(dash_only_text_ctr.values()),
        "ben_only_text_count": sum(ben_only_text_ctr.values()),
        "dash_only_texts": list(dash_only_text_ctr.elements()),
        "ben_only_texts": list(ben_only_text_ctr.elements()),

        # Step 2 outputs
        "pct_label_match_given_shared_text": pct_label_match_given_shared_text,
        "label_match_count": matched_label_instances,
        "label_comparison_count": shared_label_comparison_instances,
        "label_mismatch_examples": label_mismatch_examples,

        # Distinct strict metric
        "pct_strict_overall_text_plus_label": pct_strict_overall
    }



In [ ]:
def build_id_map(folder: Path):
    out = {}
    for p in folder.glob("*.json"):
        tid = text_id_from_stem(p.stem)
        if tid:
            out[tid] = p
    return out

ben_map = build_id_map(BEN_DIR)
dash_map = build_id_map(DASH_DIR)

common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))
print("Matched IDs:", common_ids)

results = []
for tid in common_ids:
    res = summarize_pair(ben_map[tid], dash_map[tid])
    results.append((tid, res))

Matched IDs: ['11', '84', '514', '840', '883', '2046', '6124', '17862', '38830', '40814', '41646', '54683']


In [ ]:


# Summary table
print("\nid\tstep1_same_texts%\tstep2_label_match%|shared\tstrict_text+label%\tdash_only\tben_only")
for tid, r in results:
    print(
        f"{tid}\t"
        f"{r['pct_same_tagged_texts']:.2f}\t"
        f"{r['pct_label_match_given_shared_text']:.2f}\t"
        f"{r['pct_strict_overall_text_plus_label']:.2f}\t"
        f"{r['dash_only_text_count']}\t"
        f"{r['ben_only_text_count']}"
    )




id	step1_same_texts%	step2_label_match%|shared	strict_text+label%	dash_only	ben_only
11	92.86	96.15	86.21	4	4
84	94.44	100.00	94.44	0	1
514	82.98	79.49	56.36	7	1
840	98.00	89.80	80.00	1	0
883	97.01	100.00	97.01	1	1
2046	86.54	97.78	83.02	6	1
6124	77.22	100.00	77.22	11	7
17862	100.00	100.00	100.00	0	0
38830	87.50	95.92	81.03	4	3
40814	90.91	98.57	88.46	5	2
41646	86.67	100.00	86.67	2	4
54683	77.55	89.47	64.15	4	7


In [ ]:
import pandas as pd

summary_df = pd.DataFrame(
    [
        {
            "id": int(tid),
            "step1_same_texts_pct": r["pct_same_tagged_texts"],
            "step2_label_match_pct_shared": r["pct_label_match_given_shared_text"],
            "strict_text_plus_label_pct": r["pct_strict_overall_text_plus_label"],
            "dash_only": r["dash_only_text_count"],
            "ben_only": r["ben_only_text_count"],
        }
        for tid, r in results
    ]
).sort_values("id").reset_index(drop=True)

# Optional: rounded display version
summary_df_display = summary_df.copy()
for c in ["step1_same_texts_pct", "step2_label_match_pct_shared", "strict_text_plus_label_pct"]:
    summary_df_display[c] = summary_df_display[c].round(2)

summary_df_display

,id,step1_same_texts_pct,step2_label_match_pct_shared,strict_text_plus_label_pct,dash_only,ben_only
0,11,92.86,96.15,86.21,4,4
1,84,94.44,100.00,94.44,0,1
2,514,82.98,79.49,56.36,7,1
3,840,98.00,89.80,80.00,1,0
4,883,97.01,100.00,97.01,1,1
5,2046,86.54,97.78,83.02,6,1
6,6124,77.22,100.00,77.22,11,7
7,17862,100.00,100.00,100.00,0,0
8,38830,87.50,95.92,81.03,4,3
9,40814,90.91,98.57,88.46,5,2


In [ ]:
# Per-text details
for tid, r in results:
    print("\n" + "="*90)
    print(f"TEXT {tid}")
    print(f"Ben:  {r['ben_file']}")
    print(f"Dash: {r['dash_file']}")
    print(f"Step 1 same texts: {r['pct_same_tagged_texts']:.2f}% ({r['shared_text_count']}/{r['union_text_count']})")
    print(f"Step 2 label match on shared texts: {r['pct_label_match_given_shared_text']:.2f}% "
          f"({r['label_match_count']}/{r['label_comparison_count']})")
    print(f"Strict overall (text+label): {r['pct_strict_overall_text_plus_label']:.2f}%")

    print(f"\nDash-only texts ({r['dash_only_text_count']}):")
    for t in r["dash_only_texts"][:20]:
        print("-", t[:180])

    print(f"\nBen-only texts ({r['ben_only_text_count']}):")
    for t in r["ben_only_texts"][:20]:
        print("-", t[:180])

    print("\nLabel mismatches on shared texts (first 10):")
    for mm in r["label_mismatch_examples"][:10]:
        print("-", mm["text"])
        print("  Dash extra label sets:", mm["dash_extra_labels"])
        print("  Ben extra label sets:", mm["ben_extra_labels"])


TEXT 11
Ben:  11_manual_sample_annotations.json
Dash: 11_manual_sample_annotations (1).json
Step 1 same texts: 92.86% (104/112)
Step 2 label match on shared texts: 96.15% (100/104)
Strict overall (text+label): 86.21%

Dash-only texts (4):
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale! “How cheerfully he seems to grin, How neatly spread his claws, And w
- ‘Who am I then? Tell me that first, and then, if I like being that person, I’ll come up: if not, I’ll stay down here till I’m somebody else’
- ‘it’
- “One, two, three, and away,”

Ben-only texts (4):
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale!
- “How cheerfully he seems to grin, How neatly spread his claws, And welcome little fishes in With gently smiling jaws!”
- “Sit down, all of you, and listen to me! I’ll soon make you dry enough!”
- it

Label mismatches on shared texts (first 10):
- ‘How doth th

## Dealing with string breaks and minor differences

In [ ]:
# Cell 1: helpers for split spans, edge-char differences, and partial label overlap

from collections import defaultdict

QUOTE_OR_EDGE_CHARS = {
    '"', "'", "“", "”", "‘", "’", "«", "»",
    "(", ")", "[", "]", "{", "}", ",", ".", ";", ":", "!", "?", "-", "—"
}


def overlap_len(a, b):
    s = max(a["start"], b["start"])
    e = min(a["end"], b["end"])
    return max(0, e - s)


def merge_intervals(intervals):
    if not intervals:
        return []

    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [list(intervals[0])]

    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    return [(s, e) for s, e in merged]


def covered_len(target_span, other_spans):
    s0, e0 = target_span["start"], target_span["end"]
    pieces = []

    for sp in other_spans:
        s = max(s0, sp["start"])
        e = min(e0, sp["end"])
        if s < e:
            pieces.append((s, e))

    return sum(e - s for s, e in merge_intervals(pieces))


def is_fully_covered_by_many(target_span, other_spans, min_parts=2):
    tlen = max(0, target_span["end"] - target_span["start"])
    if tlen == 0:
        return False, []

    overlapping = [sp for sp in other_spans if overlap_len(target_span, sp) > 0]
    c_len = covered_len(target_span, overlapping)

    return (c_len == tlen and len(overlapping) >= min_parts), overlapping


def strip_one_edge_char(s):
    s = norm_text(s or "")
    if not s:
        return s

    if s[0] in QUOTE_OR_EDGE_CHARS:
        return s[1:]
    if s[-1] in QUOTE_OR_EDGE_CHARS:
        return s[:-1]

    return s


def boundary_or_quote_near_match(a, b):
    a = norm_text(a or "")
    b = norm_text(b or "")

    if a == b:
        return False, None

    # edge quote/punct difference
    if strip_one_edge_char(a) == b or a == strip_one_edge_char(b):
        return True, "edge_quote_or_punct"

    # one extra/missing char at start or end
    if len(a) + 1 == len(b) and (b[1:] == a or b[:-1] == a):
        return True, "one_char_edge"
    if len(b) + 1 == len(a) and (a[1:] == b or a[:-1] == b):
        return True, "one_char_edge"

    return False, None


def label_relation(labels_a, labels_b):
    sa, sb = set(labels_a), set(labels_b)

    if sa == sb:
        return "exact"

    if sa & sb:
        return "partial_overlap"

    return "none"


def unmatched_by_text_only(ben_spans, dash_spans):
    ben_idx = defaultdict(list)
    dash_idx = defaultdict(list)

    for i, sp in enumerate(ben_spans):
        ben_idx[sp["text_key"]].append(i)
    for j, sp in enumerate(dash_spans):
        dash_idx[sp["text_key"]].append(j)

    ben_unmatched = set(range(len(ben_spans)))
    dash_unmatched = set(range(len(dash_spans)))

    for t in set(ben_idx) & set(dash_idx):
        k = min(len(ben_idx[t]), len(dash_idx[t]))
        for i in ben_idx[t][:k]:
            ben_unmatched.discard(i)
        for j in dash_idx[t][:k]:
            dash_unmatched.discard(j)

    return ben_unmatched, dash_unmatched

In [ ]:
# Replace your analyze_relaxed_mismatches_for_tid cell with this version

def label_overlap_and_mismatch_size(labels_a, labels_b):
    a = set(labels_a)
    b = set(labels_b)
    overlap_any = int(len(a & b) > 0)
    mismatch_size = len(a.symmetric_difference(b))
    return overlap_any, mismatch_size

def covered_by_multiple_parts(target_span, candidate_spans, min_parts=2):
    overlaps = [sp for sp in candidate_spans if overlap_len(target_span, sp) > 0]
    if len(overlaps) < min_parts:
        return False, overlaps

    c_len = covered_len(target_span, overlaps)
    t_len = max(0, target_span["end"] - target_span["start"])
    return (c_len == t_len), overlaps


def global_split_merge_records(tid, ben_spans, dash_spans):
    recs = []

    # Dash span covered by multiple Ben spans (global, not just unmatched)
    for d in dash_spans:
        ok, parts = covered_by_multiple_parts(d, ben_spans, min_parts=2)
        if ok:
            union_labels = set()
            for b in parts:
                union_labels.update(b["labels_key"])

            overlap_any, mismatch_size = label_overlap_and_mismatch_size(
                d["labels_key"], union_labels
            )

            recs.append(
                {
                    "id": int(tid),
                    "type": "split_or_merge_global",
                    "side": "dash_span_covered_by_multiple_ben_spans",
                    "anchor_start": d["start"],
                    "anchor_end": d["end"],
                    "ben_labels": tuple(sorted(union_labels)),
                    "dash_labels": tuple(sorted(d["labels_key"])),
                    "one_label_overlap": overlap_any,
                    "label_mismatch_size": mismatch_size,
                    "ben_text": "",
                    "dash_text": d["raw_text"][:220],
                    "note": f"overlap_parts={len(parts)}",
                }
            )

    # Ben span covered by multiple Dash spans (global)
    for b in ben_spans:
        ok, parts = covered_by_multiple_parts(b, dash_spans, min_parts=2)
        if ok:
            union_labels = set()
            for d in parts:
                union_labels.update(d["labels_key"])

            overlap_any, mismatch_size = label_overlap_and_mismatch_size(
                b["labels_key"], union_labels
            )

            recs.append(
                {
                    "id": int(tid),
                    "type": "split_or_merge_global",
                    "side": "ben_span_covered_by_multiple_dash_spans",
                    "anchor_start": b["start"],
                    "anchor_end": b["end"],
                    "ben_labels": tuple(sorted(b["labels_key"])),
                    "dash_labels": tuple(sorted(union_labels)),
                    "one_label_overlap": overlap_any,
                    "label_mismatch_size": mismatch_size,
                    "ben_text": b["raw_text"][:220],
                    "dash_text": "",
                    "note": f"overlap_parts={len(parts)}",
                }
            )

    # dedupe by anchor + side
    seen = set()
    out = []
    for r in recs:
        k = (r["id"], r["side"], r["anchor_start"], r["anchor_end"])
        if k not in seen:
            seen.add(k)
            out.append(r)

    return out


def analyze_relaxed_mismatches_for_tid(tid, ben_file, dash_file):
    ben = load_spans(ben_file)
    dash = load_spans(dash_file)

    ben_unmatched_idx, dash_unmatched_idx = unmatched_by_text_only(ben, dash)
    ben_unmatched = [ben[i] for i in sorted(ben_unmatched_idx)]
    dash_unmatched = [dash[j] for j in sorted(dash_unmatched_idx)]

    records = global_split_merge_records(tid, ben, dash)

    used_ben = set()
    used_dash = set()

    # A) split/merge
    for i, b in enumerate(ben_unmatched):
        ok, overlaps = is_fully_covered_by_many(b, dash, min_parts=2)
        if ok:
            union_labels = set()
            for d in overlaps:
                union_labels.update(d["labels_key"])

            overlap_any, mismatch_size = label_overlap_and_mismatch_size(
                b["labels_key"], union_labels
            )

            records.append(
                {
                    "id": int(tid),
                    "type": "split_or_merge",
                    "side": "ben_span_covered_by_multiple_dash_spans",
                    "ben_labels": tuple(sorted(b["labels_key"])),
                    "dash_labels": tuple(sorted(union_labels)),
                    "one_label_overlap": overlap_any,
                    "label_mismatch_size": mismatch_size,
                    "ben_text": b["raw_text"][:220],
                    "dash_text": "",
                    "note": f"overlap_parts={len(overlaps)}",
                }
            )
            used_ben.add(i)

    for j, d in enumerate(dash_unmatched):
        ok, overlaps = is_fully_covered_by_many(d, ben, min_parts=2)
        if ok:
            union_labels = set()
            for b in overlaps:
                union_labels.update(b["labels_key"])

            overlap_any, mismatch_size = label_overlap_and_mismatch_size(
                d["labels_key"], union_labels
            )

            records.append(
                {
                    "id": int(tid),
                    "type": "split_or_merge",
                    "side": "dash_span_covered_by_multiple_ben_spans",
                    "ben_labels": tuple(sorted(union_labels)),
                    "dash_labels": tuple(sorted(d["labels_key"])),
                    "one_label_overlap": overlap_any,
                    "label_mismatch_size": mismatch_size,
                    "ben_text": "",
                    "dash_text": d["raw_text"][:220],
                    "note": f"overlap_parts={len(overlaps)}",
                }
            )
            used_dash.add(j)

    # B) pair remaining unmatched by max overlap
    ben_pool = [(i, b) for i, b in enumerate(ben_unmatched) if i not in used_ben]
    dash_pool = [(j, d) for j, d in enumerate(dash_unmatched) if j not in used_dash]

    candidates = []
    for i, b in ben_pool:
        for j, d in dash_pool:
            ov = overlap_len(b, d)
            if ov > 0:
                candidates.append((ov, i, j, b, d))
    candidates.sort(reverse=True, key=lambda x: x[0])

    paired_ben = set()
    paired_dash = set()

    for ov, i, j, b, d in candidates:
        if i in paired_ben or j in paired_dash:
            continue

        near, near_reason = boundary_or_quote_near_match(b["raw_text"], d["raw_text"])
        mtype = "boundary_or_quote_near_match" if near else "overlap_other_text_diff"

        overlap_any, mismatch_size = label_overlap_and_mismatch_size(
            b["labels_key"], d["labels_key"]
        )

        records.append(
            {
                "id": int(tid),
                "type": mtype,
                "side": "paired_overlap",
                "ben_labels": tuple(sorted(b["labels_key"])),
                "dash_labels": tuple(sorted(d["labels_key"])),
                "one_label_overlap": overlap_any,
                "label_mismatch_size": mismatch_size,
                "ben_text": b["raw_text"][:220],
                "dash_text": d["raw_text"][:220],
                "note": f"{near_reason or 'text_diff'}; overlap_chars={ov}",
            }
        )

        paired_ben.add(i)
        paired_dash.add(j)

    # C) unresolved
    for i, b in ben_pool:
        if i not in paired_ben:
            records.append(
                {
                    "id": int(tid),
                    "type": "unresolved_unmatched",
                    "side": "ben_only_residual",
                    "ben_labels": tuple(sorted(b["labels_key"])),
                    "dash_labels": tuple(),
                    "one_label_overlap": 0,
                    "label_mismatch_size": len(set(b["labels_key"])),
                    "ben_text": b["raw_text"][:220],
                    "dash_text": "",
                    "note": "no overlapping unmatched dash span",
                }
            )

    for j, d in dash_pool:
        if j not in paired_dash:
            records.append(
                {
                    "id": int(tid),
                    "type": "unresolved_unmatched",
                    "side": "dash_only_residual",
                    "ben_labels": tuple(),
                    "dash_labels": tuple(sorted(d["labels_key"])),
                    "one_label_overlap": 0,
                    "label_mismatch_size": len(set(d["labels_key"])),
                    "ben_text": "",
                    "dash_text": d["raw_text"][:220],
                    "note": "no overlapping unmatched ben span",
                }
            )

    return records

In [ ]:
# Cell 3: run analysis + output tables (updated for one_label_overlap + label_mismatch_size)

relaxed_records = []

for tid in common_ids:
    relaxed_records.extend(
        analyze_relaxed_mismatches_for_tid(
            tid=tid,
            ben_file=ben_map[tid],
            dash_file=dash_map[tid],
        )
    )

relaxed_df = pd.DataFrame(relaxed_records)

# Quick sanity check
print("Columns:", list(relaxed_df.columns))
print("Rows:", len(relaxed_df))

if relaxed_df.empty:
    relaxed_summary_df = pd.DataFrame(
        columns=[
            "id",
            "split_merge_cases",
            "boundary_quote_cases",
            "other_overlap_text_diff",
            "one_label_overlap_cases",
            "unresolved_cases",
            "total_records",
        ]
    )
else:
    base_summary = (
        relaxed_df.groupby("id")
        .agg(
            split_merge_cases=("type", lambda s: (s == "split_or_merge").sum()),
            boundary_quote_cases=("type", lambda s: (s == "boundary_or_quote_near_match").sum()),
            other_overlap_text_diff=("type", lambda s: (s == "overlap_other_text_diff").sum()),
            one_label_overlap_cases=("one_label_overlap", "sum"),
            unresolved_cases=("type", lambda s: (s == "unresolved_unmatched").sum()),
            total_records=("type", "count"),
        )
        .reset_index()
    )

    # Add mismatch-size columns dynamically: label_mismatch_1_cases, label_mismatch_2_cases, ...
    mismatch_counts = (
        relaxed_df[relaxed_df["label_mismatch_size"] >= 1]
        .groupby(["id", "label_mismatch_size"])
        .size()
        .unstack(fill_value=0)
    )

    mismatch_counts.columns = [
        f"label_mismatch_{int(k)}_cases" for k in mismatch_counts.columns
    ]

    relaxed_summary_df = (
        base_summary
        .merge(mismatch_counts.reset_index(), on="id", how="left")
        .fillna(0)
        .sort_values("id")
        .reset_index(drop=True)
    )

    # Cast count-like columns to int
    for c in relaxed_summary_df.columns:
        if c != "id":
            relaxed_summary_df[c] = relaxed_summary_df[c].astype(int)

print("Relaxed mismatch summary")
relaxed_summary_df

Columns: ['id', 'type', 'side', 'anchor_start', 'anchor_end', 'ben_labels', 'dash_labels', 'one_label_overlap', 'label_mismatch_size', 'ben_text', 'dash_text', 'note']
Rows: 130
Relaxed mismatch summary


,id,split_merge_cases,boundary_quote_cases,other_overlap_text_diff,one_label_overlap_cases,unresolved_cases,total_records,label_mismatch_1_cases,label_mismatch_2_cases
0,11,0,0,1,18,6,24,19,3
1,84,0,0,0,17,1,18,18,0
2,514,1,0,1,20,5,25,12,4
3,840,0,0,0,1,1,2,1,0
4,883,0,1,0,1,0,1,0,0
5,2046,1,0,0,2,6,8,3,5
6,6124,0,1,2,3,12,15,12,0
7,38830,0,0,1,4,5,9,6,1
8,40814,0,0,0,2,7,9,5,2
9,41646,0,0,2,2,2,4,2,0


In [ ]:
import re
import pandas as pd


def parse_overlap_parts(note):
    if not isinstance(note, str):
        return 1
    m = re.search(r"overlap_parts=(\d+)", note)
    return int(m.group(1)) if m else 1


# Base (original) stats
base_rows = []
for tid, r in results:
    base_rows.append(
        {
            "id": int(tid),
            "shared_text_count": float(r["shared_text_count"]),
            "union_text_count": float(r["union_text_count"]),
            "dash_only": float(r["dash_only_text_count"]),
            "ben_only": float(r["ben_only_text_count"]),
            "label_match_count_exact": float(r["label_match_count"]),  # exact only
            "label_comparison_count": float(r["label_comparison_count"]),
            "strict_overlap_count": (
                (r["pct_strict_overall_text_plus_label"] / 100.0) * float(r["union_text_count"])
            ),
        }
    )

adj = pd.DataFrame(base_rows).set_index("id")

# Only these types are "resolved text equivalence" adjustments
resolved = relaxed_df[
    relaxed_df["type"].isin(["split_or_merge", "boundary_or_quote_near_match"])
].copy()

# Track new columns you requested
for col in ["one_label_overlap_count"]:
    adj[col] = 0.0

max_mismatch_k = int(resolved["label_mismatch_size"].max()) if len(resolved) else 0
for k in range(1, max_mismatch_k + 1):
    adj[f"label_mismatch_{k}_count"] = 0.0

# Apply text + exact-label corrections and count overlap/mismatch buckets
for _, row in resolved.iterrows():
    tid = int(row["id"])
    if tid not in adj.index:
        continue

    rtype = row.get("type", "")
    side = row.get("side", "")
    parts = parse_overlap_parts(row.get("note", ""))
    one_overlap = int(row.get("one_label_overlap", 0))
    mismatch_size = int(row.get("label_mismatch_size", 0))

    # Step 1 text-equivalence adjustments
    if rtype == "boundary_or_quote_near_match":
        adj.at[tid, "shared_text_count"] += 1
        adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - 1)
        adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - 1)

    elif rtype == "split_or_merge":
        adj.at[tid, "shared_text_count"] += 1
        if side == "ben_span_covered_by_multiple_dash_spans":
            adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - 1)
            adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - parts)
        elif side == "dash_span_covered_by_multiple_ben_spans":
            adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - 1)
            adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - parts)

    # Every resolved item adds one label-comparison unit
    adj.at[tid, "label_comparison_count"] += 1

    # Exact match still feeds original exact metric + strict metric
    if mismatch_size == 0:
        adj.at[tid, "label_match_count_exact"] += 1
        adj.at[tid, "strict_overlap_count"] += 1

    # New requested columns
    if one_overlap == 1:
        adj.at[tid, "one_label_overlap_count"] += 1
    if mismatch_size >= 1:
        col = f"label_mismatch_{mismatch_size}_count"
        if col in adj.columns:
            adj.at[tid, col] += 1

# Recompute metrics
adj["union_text_count"] = adj["shared_text_count"] + adj["dash_only"] + adj["ben_only"]

adj["step1_same_texts_pct"] = adj.apply(
    lambda x: (100.0 * x["shared_text_count"] / x["union_text_count"])
    if x["union_text_count"] > 0 else 100.0,
    axis=1,
)

# Keep your original exact-label metric
adj["step2_label_match_pct_shared"] = adj.apply(
    lambda x: (100.0 * x["label_match_count_exact"] / x["label_comparison_count"])
    if x["label_comparison_count"] > 0 else 100.0,
    axis=1,
)

adj["strict_text_plus_label_pct"] = adj.apply(
    lambda x: (100.0 * x["strict_overlap_count"] / x["union_text_count"])
    if x["union_text_count"] > 0 else 100.0,
    axis=1,
)

# Added rate: at least one label overlap
adj["one_label_overlap_pct_shared"] = adj.apply(
    lambda x: (100.0 * x["one_label_overlap_count"] / x["label_comparison_count"])
    if x["label_comparison_count"] > 0 else 100.0,
    axis=1,
)

# Build final table
base_cols = [
    "id",
    "step1_same_texts_pct",
    "step2_label_match_pct_shared",
    "strict_text_plus_label_pct",
    "dash_only",
    "ben_only",
    "one_label_overlap_pct_shared",
    "one_label_overlap_count",
]
mismatch_cols = [c for c in adj.columns if c.startswith("label_mismatch_") and c.endswith("_count")]
mismatch_cols = sorted(mismatch_cols, key=lambda c: int(c.split("_")[2]))

final_cols = base_cols + mismatch_cols

adjusted_summary_df = (
    adj.reset_index()[final_cols]
    .sort_values("id")
    .reset_index(drop=True)
)

# Display formatting
display_df = adjusted_summary_df.copy()
for c in ["step1_same_texts_pct", "step2_label_match_pct_shared", "strict_text_plus_label_pct", "one_label_overlap_pct_shared"]:
    display_df[c] = display_df[c].round(2)

for c in ["dash_only", "ben_only", "one_label_overlap_count"] + mismatch_cols:
    display_df[c] = display_df[c].round(0).astype(int)

display_df

,id,step1_same_texts_pct,step2_label_match_pct_shared,strict_text_plus_label_pct,dash_only,ben_only,one_label_overlap_pct_shared,one_label_overlap_count,label_mismatch_1_count
0,11,92.86,96.15,86.21,4,4,0.00,0,0
1,84,94.44,100.00,94.44,0,1,0.00,0,0
2,514,86.96,77.50,57.59,6,0,2.50,1,1
3,840,98.00,89.80,80.00,1,0,0.00,0,0
4,883,100.00,100.00,100.00,0,0,1.52,1,0
5,2046,92.00,95.65,86.34,4,0,2.17,1,1
6,6124,79.49,100.00,79.49,10,6,1.61,1,0
7,17862,100.00,100.00,100.00,0,0,0.00,0,0
8,38830,87.50,95.92,81.03,4,3,0.00,0,0
9,40814,90.91,98.57,88.46,5,2,0.00,0,0


In [ ]:
import re
import pandas as pd
from collections import Counter, defaultdict


def parse_overlap_parts(note):
    if not isinstance(note, str):
        return 1
    m = re.search(r"overlap_parts=(\d+)", note)
    return int(m.group(1)) if m else 1


def pair_mismatch_labelsets(ben_list, dash_list):
    """
    Greedy pairing for remaining non-exact labelsets.
    Returns list of (one_overlap, mismatch_size) pairs.
    """
    pairs = []
    ben = ben_list[:]
    dash = dash_list[:]

    while ben and dash:
        best_i = best_j = None
        best_key = None  # maximize overlap, then minimize mismatch size

        for i, b in enumerate(ben):
            sb = set(b)
            for j, d in enumerate(dash):
                sd = set(d)
                overlap = len(sb & sd)
                mismatch = len(sb.symmetric_difference(sd))
                key = (overlap, -mismatch)
                if best_key is None or key > best_key:
                    best_key = key
                    best_i, best_j = i, j

        b = ben.pop(best_i)
        d = dash.pop(best_j)
        sb, sd = set(b), set(d)
        one_overlap = int(len(sb & sd) > 0)
        mismatch_size = len(sb.symmetric_difference(sd))
        pairs.append((one_overlap, mismatch_size))

    return pairs


def base_step2_label_buckets(ben_file, dash_file):
    """
    Buckets over ORIGINAL Step 2 comparisons (shared text only, text_key-based),
    before relaxed corrections.
    """
    ben = load_spans(ben_file)
    dash = load_spans(dash_file)

    ben_text_ctr = Counter(r["text_key"] for r in ben)
    dash_text_ctr = Counter(r["text_key"] for r in dash)
    shared_text_ctr = ben_text_ctr & dash_text_ctr

    ben_text_to_labels = defaultdict(Counter)
    dash_text_to_labels = defaultdict(Counter)

    for r in ben:
        ben_text_to_labels[r["text_key"]][r["labels_key"]] += 1
    for r in dash:
        dash_text_to_labels[r["text_key"]][r["labels_key"]] += 1

    total = 0
    exact = 0
    one_overlap = 0
    mismatch_counts = Counter()

    for text_key, shared_n in shared_text_ctr.items():
        total += shared_n

        ben_ctr = ben_text_to_labels[text_key].copy()
        dash_ctr = dash_text_to_labels[text_key].copy()

        # Exact matches first
        common = ben_ctr & dash_ctr
        exact_n = sum(common.values())
        exact += exact_n
        one_overlap += exact_n  # exact is also one-label-overlap

        ben_ctr -= common
        dash_ctr -= common

        # Expand leftovers and pair mismatches
        ben_left = list(ben_ctr.elements())
        dash_left = list(dash_ctr.elements())

        # sanity: should be equal lengths
        if len(ben_left) != len(dash_left):
            m = min(len(ben_left), len(dash_left))
            ben_left = ben_left[:m]
            dash_left = dash_left[:m]

        for ov, mm in pair_mismatch_labelsets(ben_left, dash_left):
            if ov == 1:
                one_overlap += 1
            if mm >= 1:
                mismatch_counts[mm] += 1

    return {
        "label_comparison_count_base": float(total),
        "label_match_count_exact_base": float(exact),
        "one_label_overlap_count_base": float(one_overlap),
        "mismatch_counts_base": mismatch_counts,
    }


# ---------- Build base rows ----------
base_rows = []
for tid, r in results:
    buckets = base_step2_label_buckets(ben_map[tid], dash_map[tid])

    row = {
        "id": int(tid),
        "shared_text_count": float(r["shared_text_count"]),
        "union_text_count": float(r["union_text_count"]),
        "dash_only": float(r["dash_only_text_count"]),
        "ben_only": float(r["ben_only_text_count"]),
        "label_match_count_exact": buckets["label_match_count_exact_base"],
        "label_comparison_count": buckets["label_comparison_count_base"],
        "one_label_overlap_count": buckets["one_label_overlap_count_base"],
        "strict_overlap_count": (
            (r["pct_strict_overall_text_plus_label"] / 100.0) * float(r["union_text_count"])
        ),
    }

    # attach dynamic mismatch_k counts from base
    for k, v in buckets["mismatch_counts_base"].items():
        row[f"label_mismatch_{int(k)}_count"] = float(v)

    base_rows.append(row)

adj = pd.DataFrame(base_rows).fillna(0).set_index("id")

# ---------- Apply resolved adjustments ----------
resolved = relaxed_df[
    relaxed_df["type"].isin([
        "split_or_merge",
        "split_or_merge_global",
        "boundary_or_quote_near_match",
    ])
].copy()

# Ensure mismatch columns exist for any new k seen in resolved rows
if not resolved.empty:
    max_k_resolved = int(resolved["label_mismatch_size"].max())
    for k in range(1, max_k_resolved + 1):
        col = f"label_mismatch_{k}_count"
        if col not in adj.columns:
            adj[col] = 0.0

for _, row in resolved.iterrows():
    tid = int(row["id"])
    if tid not in adj.index:
        continue

    rtype = row.get("type", "")
    side = row.get("side", "")
    parts = parse_overlap_parts(row.get("note", ""))
    one_overlap = int(row.get("one_label_overlap", 0))
    mismatch_size = int(row.get("label_mismatch_size", 0))

    # Step 1 text-equivalence adjustments
    if rtype == "boundary_or_quote_near_match":
        adj.at[tid, "shared_text_count"] += 1
        adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - 1)
        adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - 1)

    elif rtype in ("split_or_merge", "split_or_merge_global"):
        adj.at[tid, "shared_text_count"] += 1
        if side == "ben_span_covered_by_multiple_dash_spans":
            adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - 1)
            adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - parts)
        elif side == "dash_span_covered_by_multiple_ben_spans":
            adj.at[tid, "dash_only"] = max(0.0, adj.at[tid, "dash_only"] - 1)
            adj.at[tid, "ben_only"] = max(0.0, adj.at[tid, "ben_only"] - parts)

    # Step 2 denominator increases for each newly resolved text equivalence
    adj.at[tid, "label_comparison_count"] += 1

    # Exact match
    if mismatch_size == 0:
        adj.at[tid, "label_match_count_exact"] += 1
        adj.at[tid, "strict_overlap_count"] += 1

    # One-label-overlap (overall)
    if one_overlap == 1:
        adj.at[tid, "one_label_overlap_count"] += 1

    # mismatch size buckets
    if mismatch_size >= 1:
        col = f"label_mismatch_{mismatch_size}_count"
        if col not in adj.columns:
            adj[col] = 0.0
        adj.at[tid, col] += 1


# ---------- Recompute metrics ----------
adj["union_text_count"] = adj["shared_text_count"] + adj["dash_only"] + adj["ben_only"]

adj["step1_same_texts_pct"] = adj.apply(
    lambda x: (100.0 * x["shared_text_count"] / x["union_text_count"])
    if x["union_text_count"] > 0 else 100.0,
    axis=1,
)

# Exact-only (your original Step 2 definition)
adj["step2_label_match_pct_shared"] = adj.apply(
    lambda x: (100.0 * x["label_match_count_exact"] / x["label_comparison_count"])
    if x["label_comparison_count"] > 0 else 100.0,
    axis=1,
)

adj["strict_text_plus_label_pct"] = adj.apply(
    lambda x: (100.0 * x["strict_overlap_count"] / x["union_text_count"])
    if x["union_text_count"] > 0 else 100.0,
    axis=1,
)

# Overall one-label-overlap share of all Step 2 comparisons
adj["one_label_overlap_pct_shared"] = adj.apply(
    lambda x: (100.0 * x["one_label_overlap_count"] / x["label_comparison_count"])
    if x["label_comparison_count"] > 0 else 100.0,
    axis=1,
)

# Optional: mismatch-size percentages over all Step 2 comparisons
mismatch_cols = sorted(
    [c for c in adj.columns if c.startswith("label_mismatch_") and c.endswith("_count")],
    key=lambda c: int(c.split("_")[2]),
)
for c in mismatch_cols:
    pct_col = c.replace("_count", "_pct_shared")
    adj[pct_col] = adj.apply(
        lambda x: (100.0 * x[c] / x["label_comparison_count"])
        if x["label_comparison_count"] > 0 else 0.0,
        axis=1,
    )

# ---------- Final table ----------
base_cols = [
    "id",
    "step1_same_texts_pct",
    "step2_label_match_pct_shared",
    "strict_text_plus_label_pct",
    "dash_only",
    "ben_only",
    "one_label_overlap_pct_shared",
    "one_label_overlap_count",
]
pct_mismatch_cols = [c.replace("_count", "_pct_shared") for c in mismatch_cols]
final_cols = base_cols + mismatch_cols + pct_mismatch_cols

adjusted_summary_df = (
    adj.reset_index()[final_cols]
    .sort_values("id")
    .reset_index(drop=True)
)

display_df = adjusted_summary_df.copy()
for c in [
    "step1_same_texts_pct",
    "step2_label_match_pct_shared",
    "strict_text_plus_label_pct",
    "one_label_overlap_pct_shared",
] + pct_mismatch_cols:
    display_df[c] = display_df[c].round(2)

for c in ["dash_only", "ben_only", "one_label_overlap_count"] + mismatch_cols:
    display_df[c] = display_df[c].round(0).astype(int)

display_df

,id,step1_same_texts_pct,step2_label_match_pct_shared,strict_text_plus_label_pct,dash_only,ben_only,one_label_overlap_pct_shared,one_label_overlap_count,label_mismatch_1_count,label_mismatch_2_count,label_mismatch_1_pct_shared,label_mismatch_2_pct_shared
0,11,100.00,84.30,81.45,0,0,100.00,121,19,0,15.70,0.00
1,84,100.00,50.00,50.00,0,0,100.00,34,17,0,50.00,0.00
2,514,100.00,67.24,59.47,0,0,100.00,58,19,0,32.76,0.00
3,840,100.00,90.00,82.00,0,0,90.00,45,0,5,0.00,10.00
4,883,100.00,100.00,100.00,0,0,100.00,66,0,0,0.00,0.00
5,2046,95.92,93.62,88.10,2,0,97.87,46,2,1,4.26,2.13
6,6124,79.49,100.00,79.49,10,6,100.00,62,0,0,0.00,0.00
7,17862,100.00,100.00,100.00,0,0,100.00,13,0,0,0.00,0.00
8,38830,100.00,92.31,89.19,0,0,98.08,51,3,1,5.77,1.92
9,40814,98.63,98.61,96.05,1,0,98.61,71,0,1,0.00,1.39


In [ ]:
import re

# 1) Rename columns
rename_map = {
    "step1_same_texts_pct": "tagged_text_overlap_pct",
    "step2_label_match_pct_shared": "exact_label_match_pct",
    "strict_text_plus_label_pct": "strict_text_AND_label_pct",
    "dash_only": "dash_only_count",
    "ben_only": "ben_only_count",
    "one_label_overlap_pct_shared": "at_least_one_shared_label",
    "one_label_overlap_count": "any_label_overlap_count",
}
df = display_df.rename(columns=rename_map)

# 2) Keep dynamic mismatch columns in numeric order
mismatch_count_cols = sorted(
    [c for c in df.columns if re.fullmatch(r"label_mismatch_\d+_count", c)],
    key=lambda c: int(c.split("_")[2])
)
mismatch_pct_cols = sorted(
    [c for c in df.columns if re.fullmatch(r"label_mismatch_\d+_pct_shared", c)],
    key=lambda c: int(c.split("_")[2])
)

df.head()

,id,tagged_text_overlap_pct,exact_label_match_pct,strict_text_AND_label_pct,dash_only_count,ben_only_count,at_least_one_shared_label,any_label_overlap_count,label_mismatch_1_count,label_mismatch_2_count,label_mismatch_1_pct_shared,label_mismatch_2_pct_shared
0,11,100.0,84.30,81.45,0,0,100.0,121,19,0,15.70,0.0
1,84,100.0,50.00,50.00,0,0,100.0,34,17,0,50.00,0.0
2,514,100.0,67.24,59.47,0,0,100.0,58,19,0,32.76,0.0
3,840,100.0,90.00,82.00,0,0,90.0,45,0,5,0.00,10.0
4,883,100.0,100.00,100.00,0,0,100.0,66,0,0,0.00,0.0


In [ ]:
# strict exact text match % from the original strict text-key comparison (before resolution)
strict_text_pct_map = {
    int(tid): r["pct_same_tagged_texts"]
    for tid, r in results
}

df["strict_exact_text_match_pct"] = (
    pd.to_numeric(df["id"], errors="coerce")
    .map(strict_text_pct_map)
    .round(2)
)

df.head()

,id,tagged_text_overlap_pct,exact_label_match_pct,strict_text_AND_label_pct,dash_only_count,ben_only_count,at_least_one_shared_label,any_label_overlap_count,label_mismatch_1_count,label_mismatch_2_count,label_mismatch_1_pct_shared,label_mismatch_2_pct_shared,strict_exact_text_match_pct
0,11,100.0,84.30,81.45,0,0,100.0,121,19,0,15.70,0.0,92.86
1,84,100.0,50.00,50.00,0,0,100.0,34,17,0,50.00,0.0,94.44
2,514,100.0,67.24,59.47,0,0,100.0,58,19,0,32.76,0.0,82.98
3,840,100.0,90.00,82.00,0,0,90.0,45,0,5,0.00,10.0,98.00
4,883,100.0,100.00,100.00,0,0,100.0,66,0,0,0.00,0.0,97.01


In [ ]:
df.columns.values

array(['id', 'tagged_text_overlap_pct', 'exact_label_match_pct',
       'strict_text_AND_label_pct', 'dash_only_count', 'ben_only_count',
       'at_least_one_shared_label', 'any_label_overlap_count',
       'label_mismatch_1_count', 'label_mismatch_2_count',
       'label_mismatch_1_pct_shared', 'label_mismatch_2_pct_shared',
       'strict_exact_text_match_pct'], dtype=object)

In [ ]:

# 3) Build desired order
front_cols = [
    "id",
    "tagged_text_overlap_pct",
    "strict_exact_text_match_pct",
    "at_least_one_shared_label",
    "exact_label_match_pct",
    "strict_text_AND_label_pct",
    "any_label_overlap_count",
    "dash_only_count",
    "ben_only_count",
]

ordered_cols = front_cols + mismatch_count_cols + mismatch_pct_cols

# Optional: keep any extra columns at the end (instead of dropping them)
extra_cols = [c for c in df.columns if c not in ordered_cols]
df = df[ordered_cols + extra_cols]

df.head()


,id,tagged_text_overlap_pct,strict_exact_text_match_pct,at_least_one_shared_label,exact_label_match_pct,strict_text_AND_label_pct,any_label_overlap_count,dash_only_count,ben_only_count,label_mismatch_1_count,label_mismatch_2_count,label_mismatch_1_pct_shared,label_mismatch_2_pct_shared
0,11,100.0,92.86,100.0,84.30,81.45,121,0,0,19,0,15.70,0.0
1,84,100.0,94.44,100.0,50.00,50.00,34,0,0,17,0,50.00,0.0
2,514,100.0,82.98,100.0,67.24,59.47,58,0,0,19,0,32.76,0.0
3,840,100.0,98.00,90.0,90.00,82.00,45,0,0,0,5,0.00,10.0
4,883,100.0,97.01,100.0,100.00,100.00,66,0,0,0,0,0.00,0.0


In [ ]:
df.columns.values

array(['id', 'tagged_text_overlap_pct', 'strict_exact_text_match_pct',
       'at_least_one_shared_label', 'exact_label_match_pct',
       'strict_text_AND_label_pct', 'any_label_overlap_count',
       'dash_only_count', 'ben_only_count', 'label_mismatch_1_count',
       'label_mismatch_2_count', 'label_mismatch_1_pct_shared',
       'label_mismatch_2_pct_shared'], dtype=object)

In [ ]:
drop_cols = ['any_label_overlap_count', 'label_mismatch_1_count',
       'label_mismatch_2_count', 'label_mismatch_1_pct_shared',
       'label_mismatch_2_pct_shared']

df = df.drop(columns=drop_cols, errors="ignore")

df.head(12)



,id,tagged_text_overlap_pct,strict_exact_text_match_pct,at_least_one_shared_label,exact_label_match_pct,strict_text_AND_label_pct,dash_only_count,ben_only_count
0,11,100.00,92.86,100.00,84.30,81.45,0,0
1,84,100.00,94.44,100.00,50.00,50.00,0,0
2,514,100.00,82.98,100.00,67.24,59.47,0,0
3,840,100.00,98.00,90.00,90.00,82.00,0,0
4,883,100.00,97.01,100.00,100.00,100.00,0,0
5,2046,95.92,86.54,97.87,93.62,88.10,2,0
6,6124,79.49,77.22,100.00,100.00,79.49,10,6
7,17862,100.00,100.00,100.00,100.00,100.00,0,0
8,38830,100.00,87.50,98.08,92.31,89.19,0,0
9,40814,98.63,90.91,98.61,98.61,96.05,1,0


In [ ]:
# Overall (all texts combined), count-weighted

import re
import pandas as pd

# mismatch count columns available in adj
mismatch_cols = sorted(
    [c for c in adj.columns if c.startswith("label_mismatch_") and c.endswith("_count")],
    key=lambda c: int(c.split("_")[2]),
)

# totals from adj (internal names stay the same)
total_shared = float(adj["shared_text_count"].sum())
total_union = float(adj["union_text_count"].sum())
total_dash_only = float(adj["dash_only"].sum())
total_ben_only = float(adj["ben_only"].sum())

total_label_comp = float(adj["label_comparison_count"].sum())
total_label_exact = float(adj["label_match_count_exact"].sum())
total_strict_overlap = float(adj["strict_overlap_count"].sum())
total_one_overlap = float(adj["one_label_overlap_count"].sum())

# overall strict-exact text match (pre-resolution baseline, from results)
total_shared_strict = sum(r["shared_text_count"] for _, r in results)
total_union_strict = sum(r["union_text_count"] for _, r in results)
overall_strict_exact_text_match_pct = (
    100.0 * total_shared_strict / total_union_strict
    if total_union_strict > 0 else 100.0
)


In [ ]:
overall = {
    "scope": "overall_all_texts",
    "tagged_text_overlap_pct": (100.0 * total_shared / total_union) if total_union > 0 else 100.0,
    "strict_exact_text_match_pct": overall_strict_exact_text_match_pct,
    "exact_label_match_pct": (100.0 * total_label_exact / total_label_comp) if total_label_comp > 0 else 100.0,
    "strict_text_AND_label_pct": (100.0 * total_strict_overlap / total_union) if total_union > 0 else 100.0,
    "dash_only_count": total_dash_only,
    "ben_only_count": total_ben_only,
    "at_least_one_shared_label": (100.0 * total_one_overlap / total_label_comp) if total_label_comp > 0 else 100.0,
    # keep this only if you still want it in final table
    "any_label_overlap_count": total_one_overlap,
}

# add mismatch count + pct_shared
for c in mismatch_cols:
    cnt = float(adj[c].sum())
    overall[c] = cnt
    overall[c.replace("_count", "_pct_shared")] = (
        (100.0 * cnt / total_label_comp) if total_label_comp > 0 else 0.0
    )

overall_df = pd.DataFrame([overall])

In [ ]:
overall_df.head()

,scope,tagged_text_overlap_pct,strict_exact_text_match_pct,exact_label_match_pct,strict_text_AND_label_pct,dash_only_count,ben_only_count,at_least_one_shared_label,any_label_overlap_count,label_mismatch_1_count,label_mismatch_1_pct_shared,label_mismatch_2_count,label_mismatch_2_pct_shared
0,overall_all_texts,96.345029,88.571429,88.01214,82.184372,15.0,10.0,98.786039,651.0,70.0,10.622155,9.0,1.365706


In [ ]:
# order columns (updated for new schema)
mismatch_count_cols = sorted(
    [c for c in overall_df.columns if re.fullmatch(r"label_mismatch_\d+_count", c)],
    key=lambda c: int(c.split("_")[2]),
)
mismatch_pct_cols = sorted(
    [c for c in overall_df.columns if re.fullmatch(r"label_mismatch_\d+_pct_shared", c)],
    key=lambda c: int(c.split("_")[2]),
)

front_cols = [
    "scope",
    "tagged_text_overlap_pct",
    "strict_exact_text_match_pct",   # new strict baseline column
    "at_least_one_shared_label",
    "exact_label_match_pct",
    "strict_text_AND_label_pct",
    "any_label_overlap_count",       # keep if you still want this shown
    "dash_only_count",
    "ben_only_count",
]

ordered_cols = [c for c in front_cols if c in overall_df.columns] + mismatch_count_cols + mismatch_pct_cols
extra_cols = [c for c in overall_df.columns if c not in ordered_cols]
overall_df = overall_df[ordered_cols + extra_cols]

# formatting
pct_cols = [c for c in overall_df.columns if c.endswith("_pct") or c.endswith("_pct_shared")]
count_cols = [c for c in overall_df.columns if c.endswith("_count")]

overall_display_df = overall_df.copy()
for c in pct_cols:
    overall_display_df[c] = pd.to_numeric(overall_display_df[c], errors="coerce").round(2)
for c in count_cols:
    overall_display_df[c] = pd.to_numeric(overall_display_df[c], errors="coerce").round(0).astype("Int64")

overall_display_df

,scope,tagged_text_overlap_pct,strict_exact_text_match_pct,at_least_one_shared_label,exact_label_match_pct,strict_text_AND_label_pct,any_label_overlap_count,dash_only_count,ben_only_count,label_mismatch_1_count,label_mismatch_2_count,label_mismatch_1_pct_shared,label_mismatch_2_pct_shared
0,overall_all_texts,96.35,88.57,98.786039,88.01,82.18,651,15,10,70,9,10.62,1.37


In [ ]:
# Build one-row overall table first (your overall_df or overall_display_df)
overall_row = overall_display_df.copy()

# Optional: make row label obvious in the per-text table
if "id" in df.columns:
    overall_row["id"] = "ALL"

# Keep only columns in df, add missing as NaN, and preserve df order
overall_row = overall_row.reindex(columns=df.columns)

# Append
df_with_total = pd.concat([df, overall_row], ignore_index=True)

df_with_total

,id,tagged_text_overlap_pct,strict_exact_text_match_pct,at_least_one_shared_label,exact_label_match_pct,strict_text_AND_label_pct,dash_only_count,ben_only_count
0,11,100.00,92.86,100.000000,84.30,81.45,0,0
1,84,100.00,94.44,100.000000,50.00,50.00,0,0
2,514,100.00,82.98,100.000000,67.24,59.47,0,0
3,840,100.00,98.00,90.000000,90.00,82.00,0,0
4,883,100.00,97.01,100.000000,100.00,100.00,0,0
5,2046,95.92,86.54,97.870000,93.62,88.10,2,0
6,6124,79.49,77.22,100.000000,100.00,79.49,10,6
7,17862,100.00,100.00,100.000000,100.00,100.00,0,0
8,38830,100.00,87.50,98.080000,92.31,89.19,0,0
9,40814,98.63,90.91,98.610000,98.61,96.05,1,0


In [ ]:
export_df = df_with_total.copy()

num_cols = export_df.select_dtypes(include="number").columns
export_df[num_cols] = export_df[num_cols].astype(float).round(2)

export_df.to_csv("inter_annotator_agreement_1.csv", index=False, float_format="%.2f", encoding="utf-8")



In [ ]:
import pandas as pd
from collections import Counter

# ---------- Compatibility helpers ----------

def get_spans_only(path):
    """
    Supports both load_spans variants:
    - returns rows
    - returns (rows, source_file)
    """
    loaded = load_spans(path)
    if isinstance(loaded, tuple):
        return loaded[0]
    return loaded


def labels_of(sp):
    """
    Supports multiple span schemas:
    - labels_set (set)
    - labels_key (tuple/list)
    - labels (tuple/list)
    """
    if isinstance(sp, dict):
        if "labels_set" in sp and sp["labels_set"] is not None:
            return set(sp["labels_set"])
        if "labels_key" in sp and sp["labels_key"] is not None:
            return set(sp["labels_key"])
        if "labels" in sp and sp["labels"] is not None:
            return set(sp["labels"])
    return set()


def text_of(sp):
    """
    Best-effort text key for near-match checks across schema variants.
    """
    if not isinstance(sp, dict):
        return ""
    if sp.get("norm_text") is not None:
        return " ".join(str(sp["norm_text"]).split()).strip()
    if sp.get("text_key") is not None:
        return " ".join(str(sp["text_key"]).split()).strip()
    if sp.get("raw_text") is not None:
        return " ".join(str(sp["raw_text"]).split()).strip()
    return ""


def one_edit_or_less(a: str, b: str) -> bool:
    if a == b:
        return True
    la, lb = len(a), len(b)
    if abs(la - lb) > 1:
        return False

    if la == lb:
        mism = sum(c1 != c2 for c1, c2 in zip(a, b))
        return mism <= 1

    # make a the shorter
    if la > lb:
        a, b = b, a
        la, lb = lb, la

    i = j = 0
    used = False
    while i < la and j < lb:
        if a[i] == b[j]:
            i += 1
            j += 1
        else:
            if used:
                return False
            used = True
            j += 1
    return True


def overlap_len(a, b):
    a_start, a_end = a.get("start"), a.get("end")
    b_start, b_end = b.get("start"), b.get("end")
    if None in (a_start, a_end, b_start, b_end):
        return 0
    s = max(a_start, b_start)
    e = min(a_end, b_end)
    return max(0, e - s)


def simple_fallback_pairings(ben_spans, dash_spans, require_label_overlap=False):
    """
    Greedy 1:1 fallback matcher if pair_spans_for_file is unavailable/incompatible.
    """
    candidates = []
    for i, b in enumerate(ben_spans):
        for j, d in enumerate(dash_spans):
            ov = overlap_len(b, d)
            if ov <= 0:
                continue

            b_labels = labels_of(b)
            d_labels = labels_of(d)
            if require_label_overlap and len(b_labels & d_labels) == 0:
                continue

            bt = text_of(b)
            dt = text_of(d)
            text_ok = (
                bt == dt
                or one_edit_or_less(bt, dt)
                or (bt and dt and (bt in dt or dt in bt))
            )
            if not text_ok:
                continue

            candidates.append((ov, i, j))

    candidates.sort(reverse=True, key=lambda x: x[0])

    used_ben = set()
    used_dash = set()
    out = []
    for _, i, j in candidates:
        if i in used_ben or j in used_dash:
            continue
        out.append(("one_to_one_fallback", [i], [j]))
        used_ben.add(i)
        used_dash.add(j)

    return out


def get_pairings_compat(ben_spans, dash_spans):
    """
    Try current project matcher first, with signature compatibility.
    Fallback to local matcher if needed.
    """
    if "pair_spans_for_file" in globals():
        try:
            p = pair_spans_for_file(ben_spans, dash_spans)
            if isinstance(p, list):
                return p
        except TypeError:
            try:
                p = pair_spans_for_file(ben_spans, dash_spans)
                if isinstance(p, list):
                    return p
            except Exception:
                pass
        except Exception:
            pass

    return simple_fallback_pairings(ben_spans, dash_spans, require_label_overlap=False)


def normalize_pairings(pairings, ben_n, dash_n):
    """
    Keep only valid (ptype, ben_idxs, dash_idxs) triplets with in-range indices.
    """
    norm = []
    for item in pairings:
        if not (isinstance(item, (list, tuple)) and len(item) == 3):
            continue
        ptype, ben_idxs, dash_idxs = item
        if not isinstance(ben_idxs, (list, tuple)) or not isinstance(dash_idxs, (list, tuple)):
            continue

        bi = [int(i) for i in ben_idxs if isinstance(i, (int,)) and 0 <= i < ben_n]
        dj = [int(j) for j in dash_idxs if isinstance(j, (int,)) and 0 <= j < dash_n]
        if not bi or not dj:
            continue
        norm.append((ptype, bi, dj))
    return norm


# ---------- Kappa helper ----------
def cohen_kappa(a, b):
    # a and b should be equal-length sequences of categorical values
    n = min(len(a), len(b))
    if n == 0:
        return float("nan")

    a = a[:n]
    b = b[:n]

    agree = sum(x == y for x, y in zip(a, b))
    p0 = agree / n

    ca = Counter(a)
    cb = Counter(b)
    cats = set(ca) | set(cb)
    pe = sum((ca[c] / n) * (cb[c] / n) for c in cats)

    if pe == 1:
        return float("nan")
    return (p0 - pe) / (1 - pe)


# ---------- Build inputs ----------
ben_map = build_id_map(BEN_DIR)
dash_map = build_id_map(DASH_DIR)
common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

# For span kappa over resolved union items:
# matched pair -> (1,1), ben-only -> (1,0), dash-only -> (0,1)
span_ben_all = []
span_dash_all = []

# For exact all-labels kappa on matched pairs
exact_ben_all = []
exact_dash_all = []

# For pooled per-label binary kappa across matched pairs
pooled_ben_label_decisions = []
pooled_dash_label_decisions = []

# Optional plain rates
matched_overlap_any_count = 0
matched_pair_count = 0

# Diagnostics
files_with_fallback_matcher = 0
files_with_pairing_errors = 0

for tid in common_ids:
    try:
        ben_spans = get_spans_only(ben_map[tid]) or []
        dash_spans = get_spans_only(dash_map[tid]) or []

        # Ensure list-like
        if not isinstance(ben_spans, list):
            ben_spans = list(ben_spans)
        if not isinstance(dash_spans, list):
            dash_spans = list(dash_spans)

        # Try project matcher first; fallback if needed
        used_project_matcher = True
        pairings_raw = []
        if "pair_spans_for_file" in globals():
            try:
                pairings_raw = pair_spans_for_file(ben_spans, dash_spans)
            except TypeError:
                pairings_raw = pair_spans_for_file(ben_spans, dash_spans)
            except Exception:
                used_project_matcher = False
        else:
            used_project_matcher = False

        if not used_project_matcher:
            files_with_fallback_matcher += 1
            pairings_raw = simple_fallback_pairings(ben_spans, dash_spans, require_label_overlap=False)

        pairings = normalize_pairings(pairings_raw, len(ben_spans), len(dash_spans))

        matched_ben = set()
        matched_dash = set()

        # Span-level vectors
        for _, ben_idxs, dash_idxs in pairings:
            matched_ben.update(ben_idxs)
            matched_dash.update(dash_idxs)
            span_ben_all.append(1)
            span_dash_all.append(1)

        ben_only_idxs = [i for i in range(len(ben_spans)) if i not in matched_ben]
        dash_only_idxs = [j for j in range(len(dash_spans)) if j not in matched_dash]

        for _ in ben_only_idxs:
            span_ben_all.append(1)
            span_dash_all.append(0)

        for _ in dash_only_idxs:
            span_ben_all.append(0)
            span_dash_all.append(1)

        # Label-level vectors on matched pairs
        for _, ben_idxs, dash_idxs in pairings:
            ben_labels = set().union(*(labels_of(ben_spans[i]) for i in ben_idxs))
            dash_labels = set().union(*(labels_of(dash_spans[j]) for j in dash_idxs))

            ben_labelset = "|".join(sorted(map(str, ben_labels)))
            dash_labelset = "|".join(sorted(map(str, dash_labels)))
            exact_ben_all.append(ben_labelset)
            exact_dash_all.append(dash_labelset)

            label_vocab = sorted(ben_labels | dash_labels, key=lambda x: str(x))
            for lab in label_vocab:
                pooled_ben_label_decisions.append(int(lab in ben_labels))
                pooled_dash_label_decisions.append(int(lab in dash_labels))

            matched_pair_count += 1
            if len(ben_labels & dash_labels) > 0:
                matched_overlap_any_count += 1

    except Exception:
        files_with_pairing_errors += 1
        continue


# ---------- Final metrics ----------
span_kappa = cohen_kappa(span_ben_all, span_dash_all)
all_labels_exact_kappa = cohen_kappa(exact_ben_all, exact_dash_all)
at_least_one_label_pooled_kappa = cohen_kappa(
    pooled_ben_label_decisions,
    pooled_dash_label_decisions
)

text_span_agreement_rate = (
    sum(int(a == b == 1) for a, b in zip(span_ben_all, span_dash_all)) / len(span_ben_all)
    if len(span_ben_all) else float("nan")
)

at_least_one_label_rate_given_shared = (
    matched_overlap_any_count / matched_pair_count if matched_pair_count else float("nan")
)

all_labels_exact_rate_given_shared = (
    sum(int(a == b) for a, b in zip(exact_ben_all, exact_dash_all)) / len(exact_ben_all)
    if len(exact_ben_all) else float("nan")
)

kappa_summary = pd.DataFrame([{
    "span_kappa_resolved_union": span_kappa,
    "label_kappa_exact_all_labels_on_matched": all_labels_exact_kappa,
    "label_kappa_pooled_binary_across_labels": at_least_one_label_pooled_kappa,
    "text_span_agreement_rate_resolved_union": text_span_agreement_rate,
    "at_least_one_label_rate_given_shared": at_least_one_label_rate_given_shared,
    "all_labels_exact_rate_given_shared": all_labels_exact_rate_given_shared,
    "n_resolved_union_items_for_span": len(span_ben_all),
    "n_matched_pairs_for_label": matched_pair_count,
    "n_common_files": len(common_ids),
    "n_files_with_fallback_matcher": files_with_fallback_matcher,
    "n_files_with_pairing_errors_skipped": files_with_pairing_errors
}])

kappa_summary

,span_kappa_resolved_union,label_kappa_exact_all_labels_on_matched,label_kappa_pooled_binary_across_labels,text_span_agreement_rate_resolved_union,at_least_one_label_rate_given_shared,all_labels_exact_rate_given_shared,n_resolved_union_items_for_span,n_matched_pairs_for_label,n_common_files,n_files_with_fallback_matcher,n_files_with_pairing_errors_skipped
0,-0.494041,0.62121,-0.044807,0.338314,0.971609,0.92429,937,317,12,12,0


In [ ]:
import pandas as pd
from collections import Counter

# ---------- Helpers ----------

def norm_text_local(s):
    return " ".join((s or "").split()).strip()

def get_spans_only(path):
    loaded = load_spans(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded

def labels_of(sp):
    if "labels_set" in sp and sp["labels_set"] is not None:
        return set(sp["labels_set"])
    if "labels_key" in sp and sp["labels_key"] is not None:
        return set(sp["labels_key"])
    if "labels" in sp and sp["labels"] is not None:
        return set(sp["labels"])
    return set()

def cohen_kappa(a, b):
    n = min(len(a), len(b))
    if n == 0:
        return float("nan")

    a = a[:n]
    b = b[:n]

    p0 = sum(x == y for x, y in zip(a, b)) / n
    ca = Counter(a)
    cb = Counter(b)
    cats = set(ca) | set(cb)
    pe = sum((ca[c] / n) * (cb[c] / n) for c in cats)

    if pe == 1:
        return float("nan")
    return (p0 - pe) / (1 - pe)

def coerce_span_for_resolver(sp):
    # Build text fields
    raw_text = sp.get("raw_text")
    if raw_text is None:
        raw_text = sp.get("text", "")
    if raw_text is None:
        raw_text = ""

    # Prefer existing normalized text if present, else derive
    if sp.get("norm_text") is not None:
        norm_text = norm_text_local(sp.get("norm_text"))
    elif sp.get("text_key") is not None:
        norm_text = norm_text_local(sp.get("text_key"))
    else:
        norm_text = norm_text_local(raw_text)

    # Build labels in all expected variants
    label_set = labels_of(sp)
    label_tuple = tuple(sorted(label_set))

    return {
        "start": sp.get("start"),
        "end": sp.get("end"),
        "raw_text": raw_text,
        "norm_text": norm_text,
        "labels_set": set(label_set),
        "labels": label_tuple,
        "labels_key": label_tuple,
        "text_key": norm_text,
    }

def coerce_spans_for_resolver(spans):
    out = []
    for sp in spans:
        if isinstance(sp, dict):
            out.append(coerce_span_for_resolver(sp))
    return out

def get_pairings_strict(ben_spans, dash_spans):
    """
    Use project resolver only.
    No fallback matcher allowed.
    """
    if "pair_spans_for_file" not in globals():
        raise RuntimeError(
            "pair_spans_for_file is not defined in kernel. "
            "Run the resolver-definition cell first."
        )

    try:
        pairings = pair_spans_for_file(
            ben_spans, dash_spans, require_label_overlap=False
        )
    except TypeError:
        pairings = pair_spans_for_file(ben_spans, dash_spans)

    if not isinstance(pairings, list):
        raise RuntimeError("pair_spans_for_file returned a non-list object.")
    return pairings

def normalize_pairings(pairings, ben_n, dash_n):
    """
    Keep only valid (ptype, ben_idxs, dash_idxs) with in-range indices.
    """
    norm = []
    dropped = 0

    for item in pairings:
        if not (isinstance(item, (list, tuple)) and len(item) == 3):
            dropped += 1
            continue

        ptype, ben_idxs, dash_idxs = item
        if not isinstance(ben_idxs, (list, tuple)) or not isinstance(dash_idxs, (list, tuple)):
            dropped += 1
            continue

        bi = [int(i) for i in ben_idxs if isinstance(i, int) and 0 <= i < ben_n]
        dj = [int(j) for j in dash_idxs if isinstance(j, int) and 0 <= j < dash_n]

        if not bi or not dj:
            dropped += 1
            continue

        norm.append((ptype, bi, dj))

    return norm, dropped

# ---------- Data ----------

ben_map = build_id_map(BEN_DIR)
dash_map = build_id_map(DASH_DIR)
common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

span_ben_all = []
span_dash_all = []

exact_ben_all = []
exact_dash_all = []

matched_pair_count = 0
matched_overlap_any_count = 0

files_failed = []
pairings_dropped_invalid_total = 0

# ---------- Main loop ----------

for tid in common_ids:
    try:
        ben_raw = get_spans_only(ben_map[tid]) or []
        dash_raw = get_spans_only(dash_map[tid]) or []

        if not isinstance(ben_raw, list):
            ben_raw = list(ben_raw)
        if not isinstance(dash_raw, list):
            dash_raw = list(dash_raw)

        # Coerce into resolver-compatible schema
        ben_spans = coerce_spans_for_resolver(ben_raw)
        dash_spans = coerce_spans_for_resolver(dash_raw)

        pairings_raw = get_pairings_strict(ben_spans, dash_spans)
        pairings, dropped_here = normalize_pairings(
            pairings_raw, len(ben_spans), len(dash_spans)
        )
        pairings_dropped_invalid_total += dropped_here

        matched_ben = set()
        matched_dash = set()

        # Span agreement vectors over resolved union
        for _, ben_idxs, dash_idxs in pairings:
            matched_ben.update(ben_idxs)
            matched_dash.update(dash_idxs)
            span_ben_all.append(1)
            span_dash_all.append(1)

        ben_only_idxs = [i for i in range(len(ben_spans)) if i not in matched_ben]
        dash_only_idxs = [j for j in range(len(dash_spans)) if j not in matched_dash]

        for _ in ben_only_idxs:
            span_ben_all.append(1)
            span_dash_all.append(0)

        for _ in dash_only_idxs:
            span_ben_all.append(0)
            span_dash_all.append(1)

        # Label agreement on matched pairs
        for _, ben_idxs, dash_idxs in pairings:
            ben_labels = set().union(*(labels_of(ben_spans[i]) for i in ben_idxs))
            dash_labels = set().union(*(labels_of(dash_spans[j]) for j in dash_idxs))

            exact_ben_all.append("|".join(sorted(map(str, ben_labels))))
            exact_dash_all.append("|".join(sorted(map(str, dash_labels))))

            matched_pair_count += 1
            if ben_labels & dash_labels:
                matched_overlap_any_count += 1

    except Exception as e:
        files_failed.append((tid, str(e)))

# Hard fail for publication safety
if files_failed:
    raise RuntimeError(
        "Resolver failed on one or more files. No metrics reported.\n"
        + "\n".join([f"id={tid}: {msg}" for tid, msg in files_failed[:10]])
    )

# ---------- Metrics ----------

span_kappa = cohen_kappa(span_ben_all, span_dash_all)
label_exact_kappa = cohen_kappa(exact_ben_all, exact_dash_all)

text_span_agreement_rate = (
    sum(int(a == b == 1) for a, b in zip(span_ben_all, span_dash_all)) / len(span_ben_all)
    if span_ben_all else float("nan")
)

label_any_overlap_rate_given_shared = (
    matched_overlap_any_count / matched_pair_count
    if matched_pair_count else float("nan")
)

label_exact_rate_given_shared = (
    sum(int(a == b) for a, b in zip(exact_ben_all, exact_dash_all)) / len(exact_ben_all)
    if exact_ben_all else float("nan")
)

kappa_summary_pubsafe = pd.DataFrame([{
    "span_kappa_resolved_union": span_kappa,
    "text_span_agreement_rate_resolved_union": text_span_agreement_rate,
    "label_kappa_exact_all_labels_on_matched": label_exact_kappa,
    "label_exact_rate_given_shared": label_exact_rate_given_shared,
    "label_any_overlap_rate_given_shared": label_any_overlap_rate_given_shared,
    "n_resolved_union_items_for_span": len(span_ben_all),
    "n_matched_pairs_for_label": matched_pair_count,
    "n_common_files": len(common_ids),
    "n_pairings_dropped_invalid": pairings_dropped_invalid_total,
    "n_files_failed": len(files_failed)
}])

kappa_summary_pubsafe

,span_kappa_resolved_union,text_span_agreement_rate_resolved_union,label_kappa_exact_all_labels_on_matched,label_exact_rate_given_shared,label_any_overlap_rate_given_shared,n_resolved_union_items_for_span,n_matched_pairs_for_label,n_common_files,n_pairings_dropped_invalid,n_files_failed
0,-0.492092,0.340107,0.621297,0.924528,0.971698,935,318,12,0,0


## Getting data about mismatches

In [ ]:
import pandas as pd

# Cases you treated as "matched after edge-case logic"
resolved_types = ["boundary_or_quote_near_match", "split_or_merge", "split_or_merge_global"]

resolved = relaxed_df[relaxed_df["type"].isin(resolved_types)].copy()

if resolved.empty:
    print("No edge-case-resolved matches found.")
else:
    # High-level reason buckets
    def reason_bucket(r):
        t = r["type"]
        side = r.get("side", "")
        note = str(r.get("note", ""))

        if t == "boundary_or_quote_near_match":
            if "edge_quote_or_punct" in note:
                return "Boundary quote/punctuation difference"
            if "one_char_edge" in note:
                return "One-character edge difference"
            return "Boundary/quote near match (other)"

        if t in ("split_or_merge", "split_or_merge_global"):
            if side == "dash_span_covered_by_multiple_ben_spans":
                return "Split/Merge: Dash span covered by multiple Ben spans"
            if side == "ben_span_covered_by_multiple_dash_spans":
                return "Split/Merge: Ben span covered by multiple Dash spans"
            return "Split/Merge (other)"

        return "Other"

    resolved["reason"] = resolved.apply(reason_bucket, axis=1)

    # 1) Most common reasons
    reason_summary = (
        resolved["reason"]
        .value_counts()
        .rename_axis("reason")
        .reset_index(name="count")
    )
    reason_summary["pct_of_resolved"] = (100 * reason_summary["count"] / reason_summary["count"].sum()).round(2)

    print("Most common reasons strict text comparison missed, but edge-case logic matched:")
    display(reason_summary)

    # 2) Optional: by text id (which texts needed rescue most often)
    by_text = (
        resolved.groupby("id")
        .size()
        .sort_values(ascending=False)
        .rename("resolved_case_count")
        .reset_index()
    )
    print("\nTexts with most edge-case recoveries:")
    display(by_text.head(15))



Most common reasons strict text comparison missed, but edge-case logic matched:


,reason,count,pct_of_resolved
0,Split/Merge: Dash span covered by multiple Ben...,36,51.43
1,Split/Merge: Ben span covered by multiple Dash...,32,45.71
2,Boundary quote/punctuation difference,1,1.43
3,One-character edge difference,1,1.43



Texts with most edge-case recoveries:


,id,resolved_case_count
0,514,19
1,11,17
2,84,17
3,54683,7
4,38830,3
5,2046,2
6,40814,2
7,840,1
8,883,1
9,6124,1


In [ ]:
# -----------------------------
# BEFORE RESOLUTION (strict)
# -----------------------------
def _clean_text(x, n=220):
    if not isinstance(x, str):
        return ""
    t = " ".join(x.split())
    return t[:n] + ("..." if len(t) > n else "")

print("\n" + "=" * 110)
print("MISMATCHES BEFORE RESOLUTION (STRICT)")
print("=" * 110)

for tid, r in results:
    dash_only = r.get("dash_only_texts", [])
    ben_only = r.get("ben_only_texts", [])
    label_mm = r.get("label_mismatch_examples", [])

    has_any = (len(dash_only) > 0) or (len(ben_only) > 0) or (len(label_mm) > 0)
    if not has_any:
        continue

    print("\n" + "-" * 110)
    print(f"TEXT {tid}")
    print(
        f"step1_same_texts={r['shared_text_count']}/{r['union_text_count']} "
        f"({r['pct_same_tagged_texts']:.2f}%), "
        f"step2_exact_labels={r['label_match_count']}/{r['label_comparison_count']} "
        f"({r['pct_label_match_given_shared_text']:.2f}%)"
    )

    if dash_only:
        print(f"\nDash-only text spans (strict unmatched): {len(dash_only)}")
        for t in dash_only:
            print(f"- {_clean_text(t)}")

    if ben_only:
        print(f"\nBen-only text spans (strict unmatched): {len(ben_only)}")
        for t in ben_only:
            print(f"- {_clean_text(t)}")

    if label_mm:
        print(f"\nShared-text label mismatches (strict): {len(label_mm)}")
        for mm in label_mm:
            print(f"- text: {_clean_text(mm.get('text', ''))}")
            print(f"  dash_extra_labels: {mm.get('dash_extra_labels', [])}")
            print(f"  ben_extra_labels:  {mm.get('ben_extra_labels', [])}")


MISMATCHES BEFORE RESOLUTION (STRICT)

--------------------------------------------------------------------------------------------------------------
TEXT 11
step1_same_texts=104/112 (92.86%), step2_exact_labels=100/104 (96.15%)

Dash-only text spans (strict unmatched): 4
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale! “How cheerfully he seems to grin, How neatly spread his claws, And welcome little fishes in With gently smil...
- ‘Who am I then? Tell me that first, and then, if I like being that person, I’ll come up: if not, I’ll stay down here till I’m somebody else’
- ‘it’
- “One, two, three, and away,”

Ben-only text spans (strict unmatched): 4
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale!
- “How cheerfully he seems to grin, How neatly spread his claws, And welcome little fishes in With gently smiling jaws!”
- “Sit down, all of you, and listen to me

In [ ]:
# -----------------------------
# BEFORE RESOLUTION (strict)
# -----------------------------
from pathlib import Path

OUT_TXT = Path("mismatch_report.txt")

def _clean_text(x, n=220):
    if not isinstance(x, str):
        return ""
    t = " ".join(x.split())
    return t[:n] + ("..." if len(t) > n else "")

with OUT_TXT.open("w", encoding="utf-8") as f:
    def w(line=""):
        print(line)
        f.write(line + "\n")

    w("\n" + "=" * 110)
    w("MISMATCHES BEFORE EDGE-CASE RESOLUTION (STRICT)")
    w("=" * 110)

    for tid, r in results:
        dash_only = r.get("dash_only_texts", [])
        ben_only = r.get("ben_only_texts", [])
        label_mm = r.get("label_mismatch_examples", [])

        has_any = (len(dash_only) > 0) or (len(ben_only) > 0) or (len(label_mm) > 0)
        if not has_any:
            continue

        w("\n" + "-" * 110)
        w(f"TEXT {tid}")
        w(
            f"step1_same_texts={r['shared_text_count']}/{r['union_text_count']} "
            f"({r['pct_same_tagged_texts']:.2f}%), "
            f"step2_exact_labels={r['label_match_count']}/{r['label_comparison_count']} "
            f"({r['pct_label_match_given_shared_text']:.2f}%)"
        )

        if dash_only:
            w(f"\nDash-only text spans (strict unmatched): {len(dash_only)}")
            for t in dash_only:
                w(f"- {_clean_text(t)}")

        if ben_only:
            w(f"\nBen-only text spans (strict unmatched): {len(ben_only)}")
            for t in ben_only:
                w(f"- {_clean_text(t)}")

        if label_mm:
            w(f"\nShared-text label mismatches (strict): {len(label_mm)}")
            for mm in label_mm:
                w(f"- text: {_clean_text(mm.get('text', ''))}")
                w(f"  dash_extra_labels: {mm.get('dash_extra_labels', [])}")
                w(f"  ben_extra_labels:  {mm.get('ben_extra_labels', [])}")

print(f"Wrote strict section to {OUT_TXT}")


MISMATCHES BEFORE EDGE-CASE RESOLUTION (STRICT)

--------------------------------------------------------------------------------------------------------------
TEXT 11
step1_same_texts=104/112 (92.86%), step2_exact_labels=100/104 (96.15%)

Dash-only text spans (strict unmatched): 4
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale! “How cheerfully he seems to grin, How neatly spread his claws, And welcome little fishes in With gently smil...
- ‘Who am I then? Tell me that first, and then, if I like being that person, I’ll come up: if not, I’ll stay down here till I’m somebody else’
- ‘it’
- “One, two, three, and away,”

Ben-only text spans (strict unmatched): 4
- “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale!
- “How cheerfully he seems to grin, How neatly spread his claws, And welcome little fishes in With gently smiling jaws!”
- “Sit down, all of you, and li

In [ ]:
# -----------------------------
# AFTER RESOLUTION
# -----------------------------
def _fmt_labels(x):
    if x is None:
        return "[]"
    if isinstance(x, (list, tuple, set)):
        return "[" + ", ".join(map(str, x)) + "]"
    return str(x)

resolved_types = {
    "boundary_or_quote_near_match",
    "split_or_merge",
    "split_or_merge_global",
}

after_df = relaxed_df[~relaxed_df["type"].isin(resolved_types)].copy()

after_df = after_df[
    (
        (after_df["type"] == "unresolved_unmatched")
        | (after_df["label_mismatch_size"].fillna(0) >= 1)
    )
    & (after_df["type"] != "overlap_other_text_diff")
].copy()

print("\n" + "=" * 110)
print("MISMATCHES AFTER RESOLUTION")
print("=" * 110)
print(f"Rows: {len(after_df)}")

for tid, g in after_df.groupby("id", sort=True):
    print("\n" + "-" * 110)
    print(f"TEXT {tid}")

    for _, r in g.iterrows():
        print(
            f"type={r.get('type')} | side={r.get('side')} | "
            f"mismatch_size={int(r.get('label_mismatch_size', 0))} | "
            f"note={r.get('note', '')}"
        )

        ben_text = _clean_text(r.get("ben_text", ""))
        dash_text = _clean_text(r.get("dash_text", ""))

        if ben_text:
            print(f"  BEN  labels={_fmt_labels(r.get('ben_labels'))}")
            print(f"  BEN  text: {ben_text}")

        if dash_text:
            print(f"  DASH labels={_fmt_labels(r.get('dash_labels'))}")
            print(f"  DASH text: {dash_text}")


MISMATCHES AFTER RESOLUTION
Rows: 52

--------------------------------------------------------------------------------------------------------------
TEXT 11
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=2 | note=no overlapping unmatched dash span
  BEN  labels=[embedded_text, speech]
  BEN  text: “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale!
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=1 | note=no overlapping unmatched dash span
  BEN  labels=[speech]
  BEN  text: “Sit down, all of you, and listen to me! I’ll soon make you dry enough!”
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=1 | note=no overlapping unmatched dash span
  BEN  labels=[embedded_text]
  BEN  text: it
type=unresolved_unmatched | side=dash_only_residual | mismatch_size=2 | note=no overlapping unmatched ben span
  DASH labels=[other, speech]
  DASH text: ‘Who am I then? Tell me that first, a

In [ ]:
# -----------------------------
# AFTER RESOLUTION
# -----------------------------
def _fmt_labels(x):
    if x is None:
        return "[]"
    if isinstance(x, (list, tuple, set)):
        return "[" + ", ".join(map(str, x)) + "]"
    return str(x)

resolved_types = {
    "boundary_or_quote_near_match",
    "split_or_merge",
    "split_or_merge_global",
}

after_df = relaxed_df[~relaxed_df["type"].isin(resolved_types)].copy()

after_df = after_df[
    (
        (after_df["type"] == "unresolved_unmatched")
        | (after_df["label_mismatch_size"].fillna(0) >= 1)
    )
    & (after_df["type"] != "overlap_other_text_diff")
].copy()

with OUT_TXT.open("a", encoding="utf-8") as f:
    def w(line=""):
        print(line)
        f.write(line + "\n")

    w("\n" + "=" * 110)
    w("MISMATCHES AFTER RESOLUTION")
    w("=" * 110)
    w(f"Rows: {len(after_df)}")

    for tid, g in after_df.groupby("id", sort=True):
        w("\n" + "-" * 110)
        w(f"TEXT {tid}")

        for _, r in g.iterrows():
            w(
                f"type={r.get('type')} | side={r.get('side')} | "
                f"mismatch_size={int(r.get('label_mismatch_size', 0))} | "
                f"note={r.get('note', '')}"
            )

            ben_text = _clean_text(r.get("ben_text", ""))
            dash_text = _clean_text(r.get("dash_text", ""))

            if ben_text:
                w(f"  BEN  labels={_fmt_labels(r.get('ben_labels'))}")
                w(f"  BEN  text: {ben_text}")

            if dash_text:
                w(f"  DASH labels={_fmt_labels(r.get('dash_labels'))}")
                w(f"  DASH text: {dash_text}")

print(f"Appended after-resolution section to {OUT_TXT}")


MISMATCHES AFTER RESOLUTION
Rows: 52

--------------------------------------------------------------------------------------------------------------
TEXT 11
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=2 | note=no overlapping unmatched dash span
  BEN  labels=[embedded_text, speech]
  BEN  text: “How doth the little crocodile Improve his shining tail, And pour the waters of the Nile On every golden scale!
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=1 | note=no overlapping unmatched dash span
  BEN  labels=[speech]
  BEN  text: “Sit down, all of you, and listen to me! I’ll soon make you dry enough!”
type=unresolved_unmatched | side=ben_only_residual | mismatch_size=1 | note=no overlapping unmatched dash span
  BEN  labels=[embedded_text]
  BEN  text: it
type=unresolved_unmatched | side=dash_only_residual | mismatch_size=2 | note=no overlapping unmatched ben span
  DASH labels=[other, speech]
  DASH text: ‘Who am I then? Tell me that first, a

## Disagreement rates per tag

In [ ]:
import pandas as pd
from collections import Counter, defaultdict

def pair_labelsets_for_shared_text(ben_ctr, dash_ctr):
    """
    Pair Ben/Dash labelsets for the same shared text, preserving multiplicity.
    Greedy: maximize label overlap, then minimize mismatch size.
    """
    ben_left = list(ben_ctr.elements())
    dash_left = list(dash_ctr.elements())

    pairs = []
    while ben_left and dash_left:
        best_i = best_j = None
        best_key = None

        for i, b in enumerate(ben_left):
            sb = set(b)
            for j, d in enumerate(dash_left):
                sd = set(d)
                overlap = len(sb & sd)
                mismatch = len(sb.symmetric_difference(sd))
                key = (overlap, -mismatch)
                if best_key is None or key > best_key:
                    best_key = key
                    best_i, best_j = i, j

        b = ben_left.pop(best_i)
        d = dash_left.pop(best_j)
        pairs.append((set(b), set(d)))

    return pairs

# Collect label-level confusion stats across all texts
stats = defaultdict(lambda: {"n": 0, "tp": 0, "tn": 0, "fp": 0, "fn": 0, "ben_pos": 0, "dash_pos": 0})

for tid in common_ids:
    ben = load_spans(ben_map[tid])
    dash = load_spans(dash_map[tid])

    ben_text_to_labels = defaultdict(Counter)
    dash_text_to_labels = defaultdict(Counter)

    for r in ben:
        ben_text_to_labels[r["text_key"]][r["labels_key"]] += 1
    for r in dash:
        dash_text_to_labels[r["text_key"]][r["labels_key"]] += 1

    shared_texts = set(ben_text_to_labels) & set(dash_text_to_labels)

    for text_key in shared_texts:
        ben_ctr = ben_text_to_labels[text_key].copy()
        dash_ctr = dash_text_to_labels[text_key].copy()

        pairs = pair_labelsets_for_shared_text(ben_ctr, dash_ctr)

        for bset, dset in pairs:
            labels_here = bset | dset
            for lab in labels_here:
                ben_has = lab in bset
                dash_has = lab in dset

                s = stats[lab]
                s["n"] += 1
                s["ben_pos"] += int(ben_has)
                s["dash_pos"] += int(dash_has)

                if ben_has and dash_has:
                    s["tp"] += 1
                elif (not ben_has) and (not dash_has):
                    s["tn"] += 1
                elif (not ben_has) and dash_has:
                    s["fp"] += 1
                else:
                    s["fn"] += 1

# Build summary table
rows = []
for lab, s in stats.items():
    n = s["n"]
    tp, tn, fp, fn = s["tp"], s["tn"], s["fp"], s["fn"]

    agreement = (tp + tn) / n if n else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = (2 * precision * recall / (precision + recall)) if (precision == precision and recall == recall and (precision + recall) > 0) else float("nan")
    pos_agree = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float("nan")
    either_pos = tp + fp + fn

    rows.append({
        "label": lab,
        "comparisons": n,
        "ben_positive": s["ben_pos"],
        "dash_positive": s["dash_pos"],
        "either_positive": either_pos,
        "both_positive": tp,
        "dash_only": fp,
        "ben_only": fn,
        "agreement_pct": 100 * agreement,
        "positive_agreement_pct": 100 * pos_agree,
        "precision_pct": 100 * precision,
        "recall_pct": 100 * recall,
        "f1_pct": 100 * f1,
    })

label_agreement_df = pd.DataFrame(rows).sort_values(
    ["either_positive", "f1_pct"],
    ascending=[False, False]
).reset_index(drop=True)

# Optional: hide very rare labels
# label_agreement_df = label_agreement_df[label_agreement_df["either_positive"] >= 5].reset_index(drop=True)

pct_cols = [c for c in label_agreement_df.columns if c.endswith("_pct")]
label_agreement_df[pct_cols] = label_agreement_df[pct_cols].round(2)


label_agreement_df

,label,comparisons,ben_positive,dash_positive,either_positive,both_positive,dash_only,ben_only,agreement_pct,positive_agreement_pct,precision_pct,recall_pct,f1_pct
0,speech,549,544,543,549,538,5,6,98.00,98.99,99.08,98.90,98.99
1,embedded_text,41,27,38,41,24,14,3,58.54,73.85,63.16,88.89,73.85
2,other,16,12,14,16,10,4,2,62.50,76.92,71.43,83.33,76.92
3,thought,15,15,15,15,15,0,0,100.00,100.00,100.00,100.00,100.00


In [ ]:
label_agreement_df.columns.values

array(['label', 'comparisons', 'ben_positive', 'dash_positive',
       'either_positive', 'both_positive', 'dash_only', 'ben_only',
       'agreement_pct', 'positive_agreement_pct', 'precision_pct',
       'recall_pct', 'f1_pct'], dtype=object)

In [ ]:
label_agreement_df = label_agreement_df[['label', 'comparisons',
       'agreement_pct', 'positive_agreement_pct', 'precision_pct',
       'recall_pct', 'f1_pct']]

label_agreement_df.rename(columns={"comparisons": "matches"})

label_agreement_df

,label,matches,agreement_pct,positive_agreement_pct,precision_pct,recall_pct,f1_pct
0,speech,549,98.00,98.99,99.08,98.90,98.99
1,embedded_text,41,58.54,73.85,63.16,88.89,73.85
2,other,16,62.50,76.92,71.43,83.33,76.92
3,thought,15,100.00,100.00,100.00,100.00,100.00


,label,comparisons,agreement_pct,positive_agreement_pct,precision_pct,recall_pct,f1_pct
0,speech,549,98.00,98.99,99.08,98.90,98.99
1,embedded_text,41,58.54,73.85,63.16,88.89,73.85
2,other,16,62.50,76.92,71.43,83.33,76.92
3,thought,15,100.00,100.00,100.00,100.00,100.00


## Expert Review

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

BEN_DIR = Path("Ben Annotations")
DASH_DIR = Path("Dash Annotations")
TEXT_DIR = Path("ManualSamplesAll")  # source text files live here in your workspace

QUOTE_CHARS = "\"'“”‘’«»"

### Getting set with both text and one tag shared

In [ ]:


def text_id_from_stem(stem: str):
    m = re.search(r"^(\d+)_manual_sample", stem)
    return m.group(1) if m else None

def norm_ws(s: str) -> str:
    return " ".join((s or "").split()).strip()

def strip_outer_quotes_and_ws(s: str) -> str:
    x = norm_ws(s or "")
    while len(x) >= 2 and x[0] in QUOTE_CHARS and x[-1] in QUOTE_CHARS:
        x = x[1:-1].strip()
    # also tolerate one-sided outer quote
    if x and x[0] in QUOTE_CHARS:
        x = x[1:].strip()
    if x and x[-1] in QUOTE_CHARS:
        x = x[:-1].strip()
    return norm_ws(x)

def one_edit_or_less(a: str, b: str) -> bool:
    """
    True if edit distance <= 1 (insert/delete/substitute single char).
    """
    if a == b:
        return True
    la, lb = len(a), len(b)
    if abs(la - lb) > 1:
        return False

    # same length -> at most one substitution
    if la == lb:
        mism = sum(c1 != c2 for c1, c2 in zip(a, b))
        return mism <= 1

    # length differs by 1 -> at most one insertion/deletion
    if la > lb:
        a, b = b, a
        la, lb = lb, la

    i = j = 0
    used = False
    while i < la and j < lb:
        if a[i] == b[j]:
            i += 1
            j += 1
        else:
            if used:
                return False
            used = True
            j += 1
    return True

def intervals_overlap(a_start, a_end, b_start, b_end):
    return max(a_start, b_start) < min(a_end, b_end)

def covered_len(target, spans):
    ts, te = target["start"], target["end"]
    pieces = []
    for sp in spans:
        s = max(ts, sp["start"])
        e = min(te, sp["end"])
        if s < e:
            pieces.append((s, e))
    if not pieces:
        return 0
    pieces.sort()
    merged = [list(pieces[0])]
    for s, e in pieces[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return sum(e - s for s, e in merged)

def load_spans(path: Path):
    obj = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for sp in obj.get("spans", []):
        labels = tuple(sorted(set(sp.get("labels", []))))
        raw = sp.get("text", "")
        rows.append({
            "start": sp.get("start", sp.get("start_byte")),
            "end": sp.get("end", sp.get("end_byte")),
            "labels": labels,
            "labels_set": set(labels),
            "raw_text": raw,
            "norm_text": strip_outer_quotes_and_ws(raw),
        })
    return rows, obj.get("source_file", "")

def build_id_map(folder: Path):
    out = {}
    for p in folder.glob("*.json"):
        tid = text_id_from_stem(p.stem)
        if tid:
            out[tid] = p
    return out

def choose_tagged_text(ben_text: str, dash_text: str) -> str:
    """
    Prefer longer normalized text if near-match (e.g., one missing letter).
    """
    bt = strip_outer_quotes_and_ws(ben_text)
    dt = strip_outer_quotes_and_ws(dash_text)

    if bt == dt:
        return bt

    # one-letter miss or very close boundary difference -> keep longer
    if one_edit_or_less(bt, dt):
        return bt if len(bt) >= len(dt) else dt

    # containment case (one is truncated)
    if bt in dt:
        return dt
    if dt in bt:
        return bt

    # fallback: longer
    return bt if len(bt) >= len(dt) else dt

from collections import defaultdict

def _text_equiv(a: str, b: str) -> bool:
    a = strip_outer_quotes_and_ws(a or "")
    b = strip_outer_quotes_and_ws(b or "")
    return (
        a == b
        or one_edit_or_less(a, b)
        or a in b
        or b in a
    )

def pair_spans_for_file(ben_spans, dash_spans):
    """
    Returns list of matched span pairs:
    1) direct overlap + shared label + text-close
    2) split/merge Ben single covered by multiple Dash
    3) split/merge Dash single covered by multiple Ben
    4) fallback exact text + exact labelset on unmatched spans
    5) fallback text-close + at least one shared label on unmatched spans
    """
    matched = []
    used_ben = set()
    used_dash = set()

    # 1) direct one-to-one by overlap and shared label
    candidates = []
    for i, b in enumerate(ben_spans):
        for j, d in enumerate(dash_spans):
            if not intervals_overlap(b["start"], b["end"], d["start"], d["end"]):
                continue

            shared = b["labels_set"] & d["labels_set"]
            if not shared:
                continue

            btxt, dtxt = b["norm_text"], d["norm_text"]
            text_ok = (
                btxt == dtxt
                or one_edit_or_less(btxt, dtxt)
                or btxt in dtxt
                or dtxt in btxt
            )
            if not text_ok:
                continue

            ov = min(b["end"], d["end"]) - max(b["start"], d["start"])
            candidates.append((ov, i, j))

    candidates.sort(reverse=True, key=lambda x: x[0])

    for _, i, j in candidates:
        if i in used_ben or j in used_dash:
            continue
        matched.append(("one_to_one", [i], [j]))
        used_ben.add(i)
        used_dash.add(j)

    # 2) split/merge Ben single covered by multiple Dash
    for i, b in enumerate(ben_spans):
        if i in used_ben:
            continue
        overlapping_dash = [
            (j, d) for j, d in enumerate(dash_spans)
            if j not in used_dash and intervals_overlap(b["start"], b["end"], d["start"], d["end"])
        ]
        if len(overlapping_dash) < 2:
            continue

        labels_groups = defaultdict(list)
        for j, d in overlapping_dash:
            labels_groups[d["labels"]].append((j, d))

        for lab_tuple, group in labels_groups.items():
            if len(group) < 2:
                continue
            d_spans = [d for _, d in group]
            if covered_len(b, d_spans) != (b["end"] - b["start"]):
                continue
            if not (b["labels_set"] & set(lab_tuple)):
                continue

            idxs = [j for j, _ in group]
            matched.append(("ben_single_dash_split", [i], idxs))
            used_ben.add(i)
            for j in idxs:
                used_dash.add(j)
            break

    # 3) split/merge Dash single covered by multiple Ben
    for j, d in enumerate(dash_spans):
        if j in used_dash:
            continue
        overlapping_ben = [
            (i, b) for i, b in enumerate(ben_spans)
            if i not in used_ben and intervals_overlap(d["start"], d["end"], b["start"], b["end"])
        ]
        if len(overlapping_ben) < 2:
            continue

        labels_groups = defaultdict(list)
        for i, b in overlapping_ben:
            labels_groups[b["labels"]].append((i, b))

        for lab_tuple, group in labels_groups.items():
            if len(group) < 2:
                continue
            b_spans = [b for _, b in group]
            if covered_len(d, b_spans) != (d["end"] - d["start"]):
                continue
            if not (d["labels_set"] & set(lab_tuple)):
                continue

            idxs = [i for i, _ in group]
            matched.append(("dash_single_ben_split", idxs, [j]))
            used_dash.add(j)
            for i in idxs:
                used_ben.add(i)
            break

    # 4) fallback: exact normalized text + exact labelset on unmatched
    ben_buckets = defaultdict(list)
    dash_buckets = defaultdict(list)

    for i, b in enumerate(ben_spans):
        if i in used_ben:
            continue
        key = (b["norm_text"], tuple(sorted(b["labels_set"])))
        ben_buckets[key].append(i)

    for j, d in enumerate(dash_spans):
        if j in used_dash:
            continue
        key = (d["norm_text"], tuple(sorted(d["labels_set"])))
        dash_buckets[key].append(j)

    for key in (set(ben_buckets) & set(dash_buckets)):
        ben_idxs = ben_buckets[key]
        dash_idxs = dash_buckets[key]
        n = min(len(ben_idxs), len(dash_idxs))
        for k in range(n):
            i = ben_idxs[k]
            j = dash_idxs[k]
            matched.append(("text_label_fallback_exact", [i], [j]))
            used_ben.add(i)
            used_dash.add(j)

    # 5) fallback: text-close + shared label on remaining unmatched
    rem_ben = [i for i in range(len(ben_spans)) if i not in used_ben]
    rem_dash = [j for j in range(len(dash_spans)) if j not in used_dash]

    fallback_candidates = []
    for i in rem_ben:
        b = ben_spans[i]
        for j in rem_dash:
            d = dash_spans[j]
            shared = b["labels_set"] & d["labels_set"]
            if not shared:
                continue
            if not _text_equiv(b["norm_text"], d["norm_text"]):
                continue
            # Prefer exact text and more shared labels
            score = (
                int(b["norm_text"] == d["norm_text"]),
                len(shared),
                -abs((b["end"] - b["start"]) - (d["end"] - d["start"]))
            )
            fallback_candidates.append((score, i, j))

    fallback_candidates.sort(reverse=True)

    for _, i, j in fallback_candidates:
        if i in used_ben or j in used_dash:
            continue
        matched.append(("text_label_fallback_relaxed", [i], [j]))
        used_ben.add(i)
        used_dash.add(j)

    return matched

def collect_shared_tag_rows():
    ben_map = build_id_map(BEN_DIR)
    dash_map = build_id_map(DASH_DIR)
    common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

    rows = []

    for tid in common_ids:
        ben_spans, ben_source_file = load_spans(ben_map[tid])
        dash_spans, dash_source_file = load_spans(dash_map[tid])

        source_file = ben_source_file or dash_source_file or f"{tid}_manual_sample.txt"

        pairings = pair_spans_for_file(ben_spans, dash_spans)

        for ptype, ben_idxs, dash_idxs in pairings:
            ben_label_union = set().union(*(ben_spans[i]["labels_set"] for i in ben_idxs))
            dash_label_union = set().union(*(dash_spans[j]["labels_set"] for j in dash_idxs))

            shared_tags = sorted(ben_label_union & dash_label_union)
            if not shared_tags:
                continue

            ben_additional = sorted(ben_label_union - dash_label_union)
            dash_additional = sorted(dash_label_union - ben_label_union)

            ben_text_combined = " ".join(ben_spans[i]["raw_text"] for i in ben_idxs)
            dash_text_combined = " ".join(dash_spans[j]["raw_text"] for j in dash_idxs)
            tagged_text = choose_tagged_text(ben_text_combined, dash_text_combined)

            for tag in shared_tags:
                rows.append({
                    "tagged text": tagged_text,
                    "shared tag": tag,
                    "dash additional tags": dash_additional,
                    "ben additional tags": ben_additional,
                    "text file": source_file
                })

    out = pd.DataFrame(rows)

    if not out.empty:
        # make list-valued columns hashable for dedupe
        for c in ["dash additional tags", "ben additional tags"]:
            out[c] = out[c].apply(lambda x: tuple(x) if isinstance(x, list) else x)

        out = out.drop_duplicates().sort_values(
            by=["text file", "shared tag", "tagged text"],
            kind="stable"
        ).reset_index(drop=True)

        # optional: convert back to lists for display/export consistency
        for c in ["dash additional tags", "ben additional tags"]:
            out[c] = out[c].apply(lambda x: list(x) if isinstance(x, tuple) else x)

    return out


In [ ]:

shared_tagged_df = collect_shared_tag_rows()
shared_tagged_df.head(50)

,tagged text,shared tag,dash additional tags,ben additional tags,text file
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,[],[],11_manual_sample.txt
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,[],[],11_manual_sample.txt
2,Où est ma chatte?,embedded_text,[],[],11_manual_sample.txt
3,"William the Conqueror, whose cause was favoure...",embedded_text,[],[],11_manual_sample.txt
4,it,embedded_text,[],[],11_manual_sample.txt
5,it,embedded_text,[speech],[],11_manual_sample.txt
6,—found it advisable to go with Edgar Atheling ...,embedded_text,[],[],11_manual_sample.txt
7,Ahem!,speech,[],[],11_manual_sample.txt
8,Are you—are you fond—of—of dogs?,speech,[],[],11_manual_sample.txt
9,As if I would talk on such a subject! Our fami...,speech,[],[],11_manual_sample.txt


In [ ]:
shared_tagged_df.to_csv("shared_texts_tags.tsv", encoding="utf-8", sep="\t")

## Getting text tagged by both Dash and Ben buth with different labels

In [ ]:

def collect_shared_text_different_tags_rows():
    ben_map = build_id_map(BEN_DIR)
    dash_map = build_id_map(DASH_DIR)
    common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

    rows = []

    for tid in common_ids:
        ben_spans, ben_source_file = load_spans(ben_map[tid])
        dash_spans, dash_source_file = load_spans(dash_map[tid])
        source_file = ben_source_file or dash_source_file or f"{tid}_manual_sample.txt"

        # IMPORTANT: do not require shared labels here
        pairings = pair_spans_for_file(ben_spans, dash_spans)

        for ptype, ben_idxs, dash_idxs in pairings:
            ben_label_union = set().union(*(ben_spans[i]["labels_set"] for i in ben_idxs))
            dash_label_union = set().union(*(dash_spans[j]["labels_set"] for j in dash_idxs))

            shared_tags = sorted(ben_label_union & dash_label_union)

            # keep only NO-shared-label rows
            if len(shared_tags) != 0:
                continue

            # optional: enforce truly different sets (not identical)
            if ben_label_union == dash_label_union:
                continue

            dash_only_tags = sorted(dash_label_union - ben_label_union)
            ben_only_tags = sorted(ben_label_union - dash_label_union)

            ben_text_combined = " ".join(ben_spans[i]["raw_text"] for i in ben_idxs)
            dash_text_combined = " ".join(dash_spans[j]["raw_text"] for j in dash_idxs)
            tagged_text = choose_tagged_text(ben_text_combined, dash_text_combined)

            rows.append({
                "tagged text": tagged_text,
                "dash only tags": dash_only_tags,
                "ben only tags": ben_only_tags,
                "text file": source_file
            })

    out = pd.DataFrame(rows)

    if not out.empty:
        for c in ["dash only tags", "ben only tags"]:
            out[c] = out[c].apply(lambda x: tuple(x) if isinstance(x, list) else x)

        out = out.drop_duplicates().sort_values(
            by=["text file", "tagged text"],
            kind="stable"
        ).reset_index(drop=True)

        for c in ["dash only tags", "ben only tags"]:
            out[c] = out[c].apply(lambda x: list(x) if isinstance(x, tuple) else x)

    return out

In [ ]:

shared_diff_tags_df = collect_shared_text_different_tags_rows()
shared_diff_tags_df.head(50)

""


In [ ]:
shared_diff_tags_df.to_csv("shared_text_diff_tags.tsv", sep="\t", encoding="utf-8")

In [ ]:
shared_diff_tags_resolved = pd.read_csv("/Users/claudiac/Library/CloudStorage/GoogleDrive-ccarrol2@andrew.cmu.edu/Shared drives/ccarrol2 Research/dialogue/nightmare_bench/shared_text_diff_tags_resolved.txt", sep="\t", encoding="utf-8")
shared_diff_tags_resolved.head(10)

,Unnamed: 0,tagged text,dash only tags,ben only tags,text file,claudia tag
0,0,old boys,['other'],['speech'],2046_manual_sample.txt,['speech']
1,1,"Lenie,",['speech'],['other'],38830_manual_sample.txt,['speech']
2,2,Warthell Movement.,['embedded_text'],['speech'],38830_manual_sample.txt,['embedded_text']
3,3,Charges will be borne. The matter is full urgent.,['embedded_text'],['speech'],840_manual_sample.txt,['embedded_text']
4,4,It was useless to tell her; she knew what it w...,['speech'],['other'],840_manual_sample.txt,['speech']
5,5,"Ride, Ride, Ride! On His Gracious Majesty's bu...",['embedded_text'],['speech'],840_manual_sample.txt,['embedded_text']
6,6,"To our good subject, John Ridd, etc.",['embedded_text'],['speech'],840_manual_sample.txt,['embedded_text']
7,7,"by these presents, greeting. These are to requ...",NaN,['speech'],840_manual_sample.txt,['embedded_text']
8,8,"Letter 4 To Mrs. Saville, England. August 5th,...",['speech'],['embedded_text'],84_manual_sample.txt,['embedded_text']


### Getting texts only tagged by one or the other.

In [ ]:
def collect_annotator_only_rows():
    ben_map = build_id_map(BEN_DIR)
    dash_map = build_id_map(DASH_DIR)
    common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

    rows = []

    for tid in common_ids:
        ben_spans, ben_source_file = load_spans(ben_map[tid])
        dash_spans, dash_source_file = load_spans(dash_map[tid])
        source_file = ben_source_file or dash_source_file or f"{tid}_manual_sample.txt"

        # Match shared text using your resolution logic, regardless of labels
        pairings = pair_spans_for_file(ben_spans, dash_spans)

        matched_ben = set()
        matched_dash = set()
        for _, ben_idxs, dash_idxs in pairings:
            matched_ben.update(ben_idxs)
            matched_dash.update(dash_idxs)

        ben_only_idxs = [i for i in range(len(ben_spans)) if i not in matched_ben]
        dash_only_idxs = [j for j in range(len(dash_spans)) if j not in matched_dash]

        for i in ben_only_idxs:
            sp = ben_spans[i]
            rows.append({
                "tagged text": strip_outer_quotes_and_ws(sp["raw_text"]),
                "annotator": "Ben",
                "tags": sorted(sp["labels_set"]),
                "text file": source_file
            })

        for j in dash_only_idxs:
            sp = dash_spans[j]
            rows.append({
                "tagged text": strip_outer_quotes_and_ws(sp["raw_text"]),
                "annotator": "Dash",
                "tags": sorted(sp["labels_set"]),
                "text file": source_file
            })

    out = pd.DataFrame(rows)

    if not out.empty:
        # Make list-valued column hashable for dedupe
        out["tags"] = out["tags"].apply(lambda x: tuple(x) if isinstance(x, list) else x)

        out = out.drop_duplicates().sort_values(
            by=["text file", "annotator", "tagged text"],
            kind="stable"
        ).reset_index(drop=True)

        # Convert back for display/export
        out["tags"] = out["tags"].apply(lambda x: list(x) if isinstance(x, tuple) else x)

    return out

In [ ]:
one_annotator_df = collect_annotator_only_rows()
one_annotator_df.head(50)

,tagged text,annotator,tags,text file
0,How doth the little crocodile Improve his shin...,Ben,"[embedded_text, speech]",11_manual_sample.txt
1,"Sit down, all of you, and listen to me! I’ll s...",Ben,[speech],11_manual_sample.txt
2,"One, two, three, and away,",Dash,[other],11_manual_sample.txt
3,"Who am I then? Tell me that first, and then, i...",Dash,"[other, speech]",11_manual_sample.txt
4,old boys,Ben,[speech],2046_manual_sample.txt
5,Toby.,Dash,"[other, speech]",2046_manual_sample.txt
6,old boys,Dash,[other],2046_manual_sample.txt
7,slave trade.,Dash,"[embedded_text, other]",2046_manual_sample.txt
8,slave trade?,Dash,"[embedded_text, other]",2046_manual_sample.txt
9,"Lenie,",Ben,[other],38830_manual_sample.txt


In [ ]:
from collections import defaultdict, deque
import pandas as pd


def _text_close(a, b):
    a = strip_outer_quotes_and_ws(a or "")
    b = strip_outer_quotes_and_ws(b or "")
    return (
        a == b
        or one_edit_or_less(a, b)
        or a in b
        or b in a
    )


def collect_annotator_only_events(one_sided_only=True):
    ben_map = build_id_map(BEN_DIR)
    dash_map = build_id_map(DASH_DIR)
    common_ids = sorted(set(ben_map) & set(dash_map), key=lambda x: int(x))

    event_rows = []

    for tid in common_ids:
        ben_spans, ben_source_file = load_spans(ben_map[tid])
        dash_spans, dash_source_file = load_spans(dash_map[tid])
        source_file = ben_source_file or dash_source_file or f"{tid}_manual_sample.txt"

        pairings = pair_spans_for_file(
            ben_spans,
            dash_spans
        )

        matched_ben = set()
        matched_dash = set()
        for _, ben_idxs, dash_idxs in pairings:
            matched_ben.update(ben_idxs)
            matched_dash.update(dash_idxs)

        ben_only_idxs = [i for i in range(len(ben_spans)) if i not in matched_ben]
        dash_only_idxs = [j for j in range(len(dash_spans)) if j not in matched_dash]

        nodes = []

        for i in ben_only_idxs:
            sp = ben_spans[i]
            nodes.append(
                {
                    "side": "Ben",
                    "idx": i,
                    "start": sp["start"],
                    "end": sp["end"],
                    "raw_text": sp["raw_text"],
                    "labels_set": set(sp["labels_set"]),
                }
            )

        for j in dash_only_idxs:
            sp = dash_spans[j]
            nodes.append(
                {
                    "side": "Dash",
                    "idx": j,
                    "start": sp["start"],
                    "end": sp["end"],
                    "raw_text": sp["raw_text"],
                    "labels_set": set(sp["labels_set"]),
                }
            )

        n = len(nodes)
        adj = defaultdict(list)

        for a in range(n):
            for b in range(a + 1, n):
                na, nb = nodes[a], nodes[b]

                if na["side"] == nb["side"]:
                    continue

                overlap = intervals_overlap(
                    na["start"], na["end"], nb["start"], nb["end"]
                )
                near = _text_close(na["raw_text"], nb["raw_text"])

                if overlap or near:
                    adj[a].append(b)
                    adj[b].append(a)

        seen = set()

        for s in range(n):
            if s in seen:
                continue

            q = deque([s])
            seen.add(s)
            comp_nodes = []

            while q:
                u = q.popleft()
                comp_nodes.append(nodes[u])

                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        q.append(v)

            ben_nodes = [x for x in comp_nodes if x["side"] == "Ben"]
            dash_nodes = [x for x in comp_nodes if x["side"] == "Dash"]

            if one_sided_only and ben_nodes and dash_nodes:
                continue

            ben_text = " ".join(x["raw_text"] for x in ben_nodes).strip()
            dash_text = " ".join(x["raw_text"] for x in dash_nodes).strip()

            if ben_text and dash_text:
                tagged_text = choose_tagged_text(ben_text, dash_text)
            elif ben_text:
                tagged_text = strip_outer_quotes_and_ws(ben_text)
            else:
                tagged_text = strip_outer_quotes_and_ws(dash_text)

            ben_tags = sorted(
                set().union(*(x["labels_set"] for x in ben_nodes)) if ben_nodes else set()
            )
            dash_tags = sorted(
                set().union(*(x["labels_set"] for x in dash_nodes)) if dash_nodes else set()
            )

            if ben_nodes and not dash_nodes:
                event_type = "ben_only_event"
            elif dash_nodes and not ben_nodes:
                event_type = "dash_only_event"
            else:
                event_type = "mixed_unresolved_event"

            event_rows.append(
                {
                    "id": int(tid),
                    "text file": source_file,
                    "event_type": event_type,
                    "tagged text": tagged_text,
                    "ben_span_count": len(ben_nodes),
                    "dash_span_count": len(dash_nodes),
                    "ben_tags_union": ben_tags,
                    "dash_tags_union": dash_tags,
                }
            )

    out = pd.DataFrame(event_rows)

    if not out.empty:
        for c in ["ben_tags_union", "dash_tags_union"]:
            out[c] = out[c].apply(
                lambda x: tuple(x) if isinstance(x, list) else x
            )

        out = (
            out.drop_duplicates()
            .sort_values(
                by=["text file", "id", "event_type", "tagged text"],
                kind="stable",
            )
            .reset_index(drop=True)
        )

        for c in ["ben_tags_union", "dash_tags_union"]:
            out[c] = out[c].apply(
                lambda x: list(x) if isinstance(x, tuple) else x
            )

    return out

In [ ]:
one_annotator_events_df = collect_annotator_only_events(one_sided_only=True)
one_annotator_events_df.head(50)


,id,text file,event_type,tagged text,ben_span_count,dash_span_count,ben_tags_union,dash_tags_union
0,11,11_manual_sample.txt,ben_only_event,How doth the little crocodile Improve his shin...,1,0,"[embedded_text, speech]",[]
1,11,11_manual_sample.txt,ben_only_event,"Sit down, all of you, and listen to me! I’ll s...",1,0,[speech],[]
2,11,11_manual_sample.txt,dash_only_event,"One, two, three, and away,",0,1,[],[other]
3,11,11_manual_sample.txt,dash_only_event,"Who am I then? Tell me that first, and then, i...",0,1,[],"[other, speech]"
4,2046,2046_manual_sample.txt,dash_only_event,Toby.,0,1,[],"[other, speech]"
5,2046,2046_manual_sample.txt,dash_only_event,slave trade.,0,1,[],"[embedded_text, other]"
6,2046,2046_manual_sample.txt,dash_only_event,slave trade?,0,1,[],"[embedded_text, other]"
7,38830,38830_manual_sample.txt,ben_only_event,[Pg 138],1,0,[other],[]
8,38830,38830_manual_sample.txt,dash_only_event,The United States of America is a government r...,0,1,[],[embedded_text]
9,38830,38830_manual_sample.txt,dash_only_event,pillar ter pos,0,1,[],"[other, speech]"


In [ ]:
one_annotator_events_df.to_csv("one_annotator_texts.tsv", sep="\t", encoding="utf-8")

In [ ]:
one_annotator_events_df = pd.read_csv("/Users/claudiac/Library/CloudStorage/GoogleDrive-ccarrol2@andrew.cmu.edu/Shared drives/ccarrol2 Research/dialogue/nightmare_bench/one_annotator_texts_resolved.txt", sep="\t", encoding="utf-8")
one_annotator_events_df.head(50)

,Unnamed: 0,id,text file,event_type,tagged text,ben_span_count,dash_span_count,ben_tags_union,dash_tags_union,claudia tag
0,0,11,11_manual_sample.txt,dash_only_event,"One, two, three, and away,",0,1,[],['other'],['other']
1,25,84,84_manual_sample.txt,ben_only_event,"You may easily perceive, Captain Walton, that ...",1,0,['speech'],[],['speech']
2,13,514,514_manual_sample.txt,dash_only_event,"Mr. Pickwick, _Sir:_— I address you upon the s...",0,1,[],['embedded_text'],['embedded_text']
3,14,514,514_manual_sample.txt,dash_only_event,P. C.,0,1,[],"['embedded_text', 'other']",['embedded_text']
4,24,840,840_manual_sample.txt,dash_only_event,King's service,0,1,[],['speech'],['speech']
5,3,2046,2046_manual_sample.txt,dash_only_event,slave trade?,0,1,[],NaN,['other']
6,2,2046,2046_manual_sample.txt,dash_only_event,slave trade.,0,1,[],"['embedded_text', 'other']",['other']
7,1,2046,2046_manual_sample.txt,dash_only_event,Toby.,0,1,[],"['other', 'speech']",['speech']
8,21,6124,6124_manual_sample.txt,dash_only_event,"And I will add,",0,1,[],['speech'],['speech']
9,23,6124,6124_manual_sample.txt,dash_only_event,"as the first advice you have given me of late,...",0,1,[],['speech'],['speech']


In [ ]:
# If columns are already Python lists, this is enough:
shared_tagged_df["additional tags"] = (
    shared_tagged_df["dash additional tags"].fillna("").apply(lambda x: x if isinstance(x, list) else [])
    + shared_tagged_df["ben additional tags"].fillna("").apply(lambda x: x if isinstance(x, list) else [])
)

# Optional: deduplicate + sort each row
shared_tagged_df["additional tags"] = shared_tagged_df["additional tags"].apply(
    lambda tags: sorted(set(tags))
)

# Optional: drop old columns
shared_tagged_df = shared_tagged_df.drop(columns=["dash additional tags", "ben additional tags"])


In [ ]:
final_benchmark = shared_tagged_df.copy()
final_benchmark.head()

,tagged text,shared tag,text file,additional tags
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[]
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[]
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[]
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[]
4,it,embedded_text,11_manual_sample.txt,[]


In [ ]:
final_benchmark.rename(columns={"shared tag": "gold_tag"}, inplace=True)

final_benchmark.head()

,tagged text,gold_tag,text file,additional tags
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[]
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[]
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[]
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[]
4,it,embedded_text,11_manual_sample.txt,[]


In [ ]:
shared_diff_tags_resolved.rename(columns={"claudia tag": "gold_tag"}, inplace=True)

shared_diff_tags_resolved = shared_diff_tags_resolved[["tagged text", "gold_tag", "text file"]]

shared_diff_tags_resolved["gold_tag"] = (
    shared_diff_tags_resolved["gold_tag"]
    .astype(str)
    .str.extract(r"^\[\s*['\"]?(.*?)['\"]?\s*\]$", expand=False)
    .fillna(shared_diff_tags_resolved["gold_tag"])
)

shared_diff_tags_resolved.head(10)

,tagged text,gold_tag,text file
0,old boys,speech,2046_manual_sample.txt
1,"Lenie,",speech,38830_manual_sample.txt
2,Warthell Movement.,embedded_text,38830_manual_sample.txt
3,Charges will be borne. The matter is full urgent.,embedded_text,840_manual_sample.txt
4,It was useless to tell her; she knew what it w...,speech,840_manual_sample.txt
5,"Ride, Ride, Ride! On His Gracious Majesty's bu...",embedded_text,840_manual_sample.txt
6,"To our good subject, John Ridd, etc.",embedded_text,840_manual_sample.txt
7,"by these presents, greeting. These are to requ...",embedded_text,840_manual_sample.txt
8,"Letter 4 To Mrs. Saville, England. August 5th,...",embedded_text,84_manual_sample.txt


In [ ]:
final_benchmark = pd.concat([final_benchmark, shared_diff_tags_resolved], ignore_index=True)
final_benchmark.head()

,tagged text,gold_tag,text file,additional tags
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[]
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[]
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[]
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[]
4,it,embedded_text,11_manual_sample.txt,[]


In [ ]:
final_benchmark.tail()

,tagged text,gold_tag,text file,additional tags
599,It was useless to tell her; she knew what it w...,speech,840_manual_sample.txt,NaN
600,"Ride, Ride, Ride! On His Gracious Majesty's bu...",embedded_text,840_manual_sample.txt,NaN
601,"To our good subject, John Ridd, etc.",embedded_text,840_manual_sample.txt,NaN
602,"by these presents, greeting. These are to requ...",embedded_text,840_manual_sample.txt,NaN
603,"Letter 4 To Mrs. Saville, England. August 5th,...",embedded_text,84_manual_sample.txt,NaN


In [ ]:
one_annotator_events_df = pd.read_csv("/Users/claudiac/Library/CloudStorage/GoogleDrive-ccarrol2@andrew.cmu.edu/Shared drives/ccarrol2 Research/dialogue/nightmare_bench/one_annotator_texts_resolved.txt", sep="\t", encoding="utf-8")
one_annotator_events_df.head()

,Unnamed: 0,id,text file,event_type,tagged text,ben_span_count,dash_span_count,ben_tags_union,dash_tags_union,claudia tag
0,0,11,11_manual_sample.txt,dash_only_event,"One, two, three, and away,",0,1,[],['other'],['other']
1,25,84,84_manual_sample.txt,ben_only_event,"You may easily perceive, Captain Walton, that ...",1,0,['speech'],[],['speech']
2,13,514,514_manual_sample.txt,dash_only_event,"Mr. Pickwick, _Sir:_— I address you upon the s...",0,1,[],['embedded_text'],['embedded_text']
3,14,514,514_manual_sample.txt,dash_only_event,P. C.,0,1,[],"['embedded_text', 'other']",['embedded_text']
4,24,840,840_manual_sample.txt,dash_only_event,King's service,0,1,[],['speech'],['speech']


In [ ]:
one_annotator_events_df.rename(columns={"claudia tag": "gold_tag"}, inplace=True)

one_annotator_events_df = one_annotator_events_df[["tagged text", "gold_tag", "text file"]]

one_annotator_events_df["gold_tag"] = (
    one_annotator_events_df["gold_tag"]
    .astype(str)
    .str.extract(r"^\[\s*['\"]?(.*?)['\"]?\s*\]$", expand=False)
    .fillna(one_annotator_events_df["gold_tag"])
)

one_annotator_events_df.head(10)

,tagged text,gold_tag,text file
0,"One, two, three, and away,",other,11_manual_sample.txt
1,"You may easily perceive, Captain Walton, that ...",speech,84_manual_sample.txt
2,"Mr. Pickwick, _Sir:_— I address you upon the s...",embedded_text,514_manual_sample.txt
3,P. C.,embedded_text,514_manual_sample.txt
4,King's service,speech,840_manual_sample.txt
5,slave trade?,other,2046_manual_sample.txt
6,slave trade.,other,2046_manual_sample.txt
7,Toby.,speech,2046_manual_sample.txt
8,"And I will add,",speech,6124_manual_sample.txt
9,"as the first advice you have given me of late,...",speech,6124_manual_sample.txt


## Building final benchmark

In [ ]:
final_benchmark = pd.concat([final_benchmark, one_annotator_events_df], ignore_index=True)
final_benchmark.head()

,tagged text,gold_tag,text file,additional tags
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[]
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[]
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[]
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[]
4,it,embedded_text,11_manual_sample.txt,[]


In [ ]:
final_benchmark.tail()

,tagged text,gold_tag,text file,additional tags
625,and which is unhappily denied her; while her d...,NaN,54683_manual_sample.txt,NaN
626,"The daughter of the Prince, and my pupil, of w...",NaN,54683_manual_sample.txt,NaN
627,the various powers of a strong mind directed t...,other,54683_manual_sample.txt,NaN
628,"To these elegant acquirements, she unites a de...",NaN,54683_manual_sample.txt,NaN
629,"You will, I hope, pardon the liberty I am goin...",NaN,54683_manual_sample.txt,NaN


In [ ]:
final_benchmark.dropna(subset=["gold_tag"], inplace=True)
final_benchmark.reset_index(drop=True, inplace=True)

In [ ]:
final_benchmark.head()

,tagged text,gold_tag,text file,additional tags
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[]
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[]
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[]
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[]
4,it,embedded_text,11_manual_sample.txt,[]


In [ ]:
final_benchmark.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 621 entries, 0 to 620
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tagged text      621 non-null    object
 1   gold_tag         621 non-null    object
 2   text file        621 non-null    object
 3   additional tags  595 non-null    object
dtypes: object(4)
memory usage: 19.5+ KB


In [ ]:
import nltk
from nltk import word_tokenize
import os

sample_folder = "/Users/claudiac/Library/CloudStorage/GoogleDrive-ccarrol2@andrew.cmu.edu/Shared drives/ccarrol2 Research/dialogue/nightmare_bench/ManualSamplesAll"
token_count = 0

for file in os.listdir(sample_folder):
    print(file)
    file_path = os.path.join(sample_folder, file)
    text = open(file_path, encoding="utf-8").read()
    tokens = word_tokenize(text)
    wcount = len(tokens)
    print(wcount)
    token_count = token_count + wcount
    print(token_count)

print(f"Total token count: {token_count}")

84_manual_sample.txt
3402
3402
54683_manual_sample.txt
3508
6910
883_manual_sample.txt
3601
10511
6124_manual_sample.txt
3610
14121
40814_manual_sample.txt
4131
18252
2046_manual_sample.txt
3511
21763
41646_manual_sample.txt
3364
25127
840_manual_sample.txt
3325
28452
514_manual_sample.txt
3644
32096
11_manual_sample.txt
4315
36411
17862_manual_sample.txt
3508
39919
38830_manual_sample.txt
3854
43773
Total token count: 43773


In [ ]:
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

# Download tokenizer resources (safe to run repeatedly)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)  # needed in newer NLTK versions

# Ensure text values are clean strings
tagged_series = final_benchmark["tagged text"].fillna("").astype(str)

# Per-row token count
final_benchmark["tagged_token_count"] = tagged_series.apply(lambda x: len(word_tokenize(x)))

# Total tagged tokens across the whole column
total_tagged_tokens = int(final_benchmark["tagged_token_count"].sum())

print("Total tagged tokens:", total_tagged_tokens)
final_benchmark[["tagged text", "tagged_token_count"]].head()

Total tagged tokens: 23860


,tagged text,tagged_token_count
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,17
1,"Edwin and Morcar, the earls of Mercia and Nort...",29
2,Où est ma chatte?,5
3,"William the Conqueror, whose cause was favoure...",47
4,it,1


In [ ]:
# Count spans (rows). Optionally restrict to non-empty tagged text.
n_spans_all = len(final_benchmark)
n_spans_nonempty = final_benchmark["tagged text"].fillna("").astype(str).str.strip().ne("").sum()

# You already have this from the previous cell:
# total_tagged_tokens = int(final_benchmark["tagged_token_count"].sum())

rel_freq_all = (n_spans_all / total_tagged_tokens) * 1000 if total_tagged_tokens > 0 else float("nan")
rel_freq_nonempty = (n_spans_nonempty / total_tagged_tokens) * 1000 if total_tagged_tokens > 0 else float("nan")

print("Total tagged tokens:", total_tagged_tokens)
print("Number of spans (all rows):", n_spans_all)
print("Number of spans (non-empty tagged text):", n_spans_nonempty)
print("Rel. Frequency (all rows) per 1000 tokens:", round(rel_freq_all, 4))
print("Rel. Frequency (non-empty rows) per 1000 tokens:", round(rel_freq_nonempty, 4))

Total tagged tokens: 23860
Number of spans (all rows): 621
Number of spans (non-empty tagged text): 621
Rel. Frequency (all rows) per 1000 tokens: 26.0268
Rel. Frequency (non-empty rows) per 1000 tokens: 26.0268


In [ ]:
# Keep only non-empty tagged text spans
nonempty_mask = final_benchmark["tagged text"].fillna("").astype(str).str.strip().ne("")
span_lengths = final_benchmark.loc[nonempty_mask, "tagged_token_count"].astype(float)

n_spans = int(span_lengths.shape[0])
mean_span_len = span_lengths.mean()

# Sample SD (most common for reporting)
sd_span_len_sample = span_lengths.std(ddof=1)

# Optional: population SD
sd_span_len_population = span_lengths.std(ddof=0)

print("N spans:", n_spans)
print("Mean span length (tokens):", round(mean_span_len, 4))
print("SD span length (sample, ddof=1):", round(sd_span_len_sample, 4))
print("SD span length (population, ddof=0):", round(sd_span_len_population, 4))

N spans: 621
Mean span length (tokens): 38.4219
SD span length (sample, ddof=1): 154.039
SD span length (population, ddof=0): 153.9149


In [ ]:
import pandas as pd

final_benchmark["gold_tag"] = (
    final_benchmark["gold_tag"]
    .astype(str)
    .str.extract(r"^\[\s*['\"]?(.*?)['\"]?\s*\]$", expand=False)
    .fillna(final_benchmark["gold_tag"])
)

# Clean gold labels
gold = (
    final_benchmark["gold_tag"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Remove empty strings if any
gold = gold[gold.ne("")]

# Counts + percentages
tag_counts = gold.value_counts()
tag_pct = (tag_counts / tag_counts.sum() * 100).round(2)

gold_tag_summary = pd.DataFrame({
    "count": tag_counts,
    "percent": tag_pct
}).reset_index().rename(columns={"index": "gold_tag"})

print("Number of unique gold tags:", gold_tag_summary.shape[0])
gold_tag_summary

Number of unique gold tags: 4


,gold_tag,count,percent
0,speech,556,89.53
1,embedded_text,34,5.48
2,other,16,2.58
3,thought,15,2.42


In [ ]:
import pandas as pd

# Clean gold tags (same idea as your current cell)
gold = (
    final_benchmark["gold_tag"]
    .dropna()
    .astype(str)
    .str.strip()
)
gold = gold[gold.ne("")]

# Create numeric codes 1..K (K should be 4 for your case)
# If you want a fixed order, replace sorted(...) with your preferred list.
tag_order = sorted(gold.unique())
tag_to_num = {tag: i + 1 for i, tag in enumerate(tag_order)}

gold_num = gold.map(tag_to_num)

# Numeric summary
gold_tag_numeric_summary = (
    gold_num.value_counts()
    .sort_index()
    .rename_axis("gold_tag_num")
    .reset_index(name="count")
)

gold_tag_numeric_summary["percent"] = (
    100 * gold_tag_numeric_summary["count"] / gold_tag_numeric_summary["count"].sum()
).round(2)

# Optional: keep legend so you know which number means which tag
legend_df = pd.DataFrame(
    {"gold_tag_num": list(tag_to_num.values()), "gold_tag": list(tag_to_num.keys())}
).sort_values("gold_tag_num")

print("Number of unique gold tags:", len(tag_to_num))
print("\nNumeric summary:")
display(gold_tag_numeric_summary)

print("\nLegend (number -> tag):")
display(legend_df)

Number of unique gold tags: 4

Numeric summary:


,gold_tag_num,count,percent
0,1,34,5.48
1,2,16,2.58
2,3,556,89.53
3,4,15,2.42



Legend (number -> tag):


,gold_tag_num,gold_tag
0,1,embedded_text
1,2,other
2,3,speech
3,4,thought


In [ ]:
import pandas as pd
import ast

def clean_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    # unwrap simple bracketed single-label strings like "['TAG']"
    m = pd.Series([s]).str.extract(r"^\[\s*['\"]?(.*?)['\"]?\s*\]$", expand=False).iloc[0]
    s = m if isinstance(m, str) and m.strip() else s
    s = s.strip()
    return s if s else None

def parse_label_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        return [clean_label(v) for v in x if clean_label(v) is not None]

    s = str(x).strip()
    if not s or s.lower() == "nan":
        return []

    # Try parsing stringified Python/JSON-like lists
    if s.startswith("[") and s.endswith("]"):
        try:
            val = ast.literal_eval(s)
            if isinstance(val, (list, tuple, set)):
                return [clean_label(v) for v in val if clean_label(v) is not None]
        except Exception:
            pass

    # fallback: treat as single label
    c = clean_label(s)
    return [c] if c is not None else []

# Figure out which additional-label columns exist in your current final_benchmark
additional_cols = [c for c in ["additional tags", "dash additional tags", "ben additional tags"] if c in final_benchmark.columns]

all_labels = []
for _, row in final_benchmark.iterrows():
    row_labels = set()

    # gold tag
    g = clean_label(row.get("gold_tag", None))
    if g is not None:
        row_labels.add(g)

    # additional labels
    for col in additional_cols:
        row_labels.update(parse_label_list(row.get(col, None)))

    # count each label at most once per row
    all_labels.extend(sorted(row_labels))

combined_counts = pd.Series(all_labels).value_counts()
combined_summary = (
    pd.DataFrame({
        "label": combined_counts.index,
        "count": combined_counts.values
    })
    .assign(percent=lambda d: (100 * d["count"] / d["count"].sum()).round(2))
    .reset_index(drop=True)
)

print("Additional-label columns used:", additional_cols)
print("Number of unique labels:", combined_summary.shape[0])
combined_summary

Additional-label columns used: ['additional tags']
Number of unique labels: 4


,label,count,percent
0,speech,561,87.38
1,embedded_text,46,7.17
2,other,20,3.12
3,thought,15,2.34


In [ ]:
final_benchmark.head()

,tagged text,gold_tag,text file,additional tags,tagged_token_count
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[],17
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[],29
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[],5
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[],47
4,it,embedded_text,11_manual_sample.txt,[],1


In [ ]:
rows_with_additional = final_benchmark["additional tags"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
).sum()

print("Rows with >=1 additional tag:", int(rows_with_additional))

Rows with >=1 additional tag: 20


In [ ]:
final_benchmark.to_csv("final_benchmark.tsv", sep="\t", encoding="utf-8")

In [ ]:
final_benchmark.head(10)

,tagged text,gold_tag,text file,additional tags,tagged_token_count
0,A mouse—of a mouse—to a mouse—a mouse—O mouse!,embedded_text,11_manual_sample.txt,[],17
1,"Edwin and Morcar, the earls of Mercia and Nort...",embedded_text,11_manual_sample.txt,[],29
2,Où est ma chatte?,embedded_text,11_manual_sample.txt,[],5
3,"William the Conqueror, whose cause was favoure...",embedded_text,11_manual_sample.txt,[],47
4,it,embedded_text,11_manual_sample.txt,[],1
5,it,embedded_text,11_manual_sample.txt,[speech],1
6,—found it advisable to go with Edgar Atheling ...,embedded_text,11_manual_sample.txt,[],34
7,Ahem!,speech,11_manual_sample.txt,[],2
8,Are you—are you fond—of—of dogs?,speech,11_manual_sample.txt,[],12
9,As if I would talk on such a subject! Our fami...,speech,11_manual_sample.txt,[],33
